# Identifying Wireless Protocols from Baseband I/Q

### Blind feature classification, and why it needs a matched filter underneath

This notebook builds a working protocol identifier from raw complex baseband
samples, derives the mathematics behind every measurement it makes, and then
measures how well it actually performs.

It is organised around one finding. A classifier built purely from **blind
features** — bandwidths, envelope statistics, histograms of the frequency
discriminator — works well but stops working below roughly **20 dB SNR**. That
floor is not a coding deficiency. Every blind feature is a *second-order*
statistic, and second-order statistics of a signal in noise converge at a rate
no amount of cleverness improves. Adding a **preamble matched filter** — a
*first-order* statistic — moves the floor to between **&minus;6 and &minus;18 dB**
depending on preamble length, a gain of 31 to 38 dB.

## Contents

**Part I — Foundations** &nbsp;(§1–4)
signal model, synthetic ground truth, file I/O

**Part II — Blind feature extraction** &nbsp;(§5–14)
burst segmentation, spectral estimation, amplitude statistics, the FM
discriminator and its noise threshold, OFDM cyclic prefixes, chirp spread
spectrum, cyclostationary symbol-rate estimation, eye-diagram tone counting

**Part III — Classification** &nbsp;(§15–18)
feature assembly, modulation-family decision, protocol database, scoring

**Part IV — Validating the blind path** &nbsp;(§19–20)
eight synthetic protocols, multi-burst segmentation, and the SNR floor

**Part V — The preamble matched filter** &nbsp;(§21–26)
why the floor exists, detection theory, template bank, carrier-offset search,
template cross-talk, and the measured floors

**Part VI — Limitations** &nbsp;(§27)

## A note on provenance

The code cells are extracted programmatically from two tested modules,
`iq_protocol_id.py` and `iq_preamble.py`, so what you read here is what was
measured. Numerical claims in the prose are recomputed by the cells around
them — several of the constants below were originally *wrong* and the
verification cells are what caught them.

Where I cite a standard, I am working from memory of the specification rather
than from the document itself, and I have flagged the places where that
matters. In particular §22 discusses one template constant I deliberately
excluded because I do not trust my recall of it.

## 1. Signal model and notation

The input is a stream of complex samples from a quadrature receiver. A single
real passband signal at carrier $f_c$ has been mixed down and low-pass filtered
into its in-phase and quadrature components:

$$x[n] \;=\; I[n] + jQ[n] \;=\; A[n]\,e^{j\phi[n]} \;+\; w[n], \qquad t = n/f_s$$

where $f_s$ is the sample rate. The noise $w[n]$ is modelled as circularly
symmetric complex white Gaussian: real and imaginary parts independent, each
of variance $\sigma^2/2$, so $\mathbb{E}|w|^2 = \sigma^2$. Per-sample SNR is

$$\gamma \;=\; \frac{\mathbb{E}|s[n]|^2}{\mathbb{E}|w[n]|^2}.$$

Because mixing to baseband is not perfect, a residual **carrier frequency
offset** $\Delta f$ remains, appearing as a slow rotation $e^{j2\pi\Delta f t}$
across the whole record. This turns out to be the dominant practical nuisance
in Part V.

The quantities we want to recover, and the section that derives each:

| Quantity | Symbol | Section |
|---|---|---|
| Occupied bandwidth | $B$ | §6 |
| Symbol (or chip) rate | $R_s = 1/T$ | §11 |
| Modulation family | — | §16 |
| Frequency deviation, tone count | $\Delta f_{\text{dev}}$, $M$ | §12 |
| OFDM FFT size, cyclic prefix | $N$, $L$ | §9 |
| Chirp bandwidth, spreading factor | $B$, $\mathrm{SF}$ | §10 |

Two conventions used throughout. **Modulation index** for FSK is
$h = 2\Delta f_{\text{dev}}/R_s$; BLE uses $h=0.5$. **Carson's rule** gives a
rough occupied bandwidth for angle modulation,
$B \approx 2(\Delta f_{\text{dev}} + R_s/2)$, and is used only to size filters,
never as a measurement.

## 2. Setup

`QUICK` reduces trial counts in the statistical sections. The full sweeps in
Part V take roughly half an hour; the quick versions take a couple of minutes
and show the same structure with noisier floor estimates.

In [ ]:
from __future__ import annotations

import argparse
import ast
import json
import math
import sys
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy import fft as sp_fft
from scipy import signal as sps_signal
from scipy import signal as sig          # alias used by the generator module

QUICK = True          # set False to reproduce the published floor numbers

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.figsize": (11, 3.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})

RNG = np.random.default_rng(0xC0FFEE)
print(f"numpy {np.__version__}   QUICK={QUICK}")

## 3. Synthetic waveforms: ground truth first

Building the generators before the analysers is deliberate. Every measurement
in Parts II–III can then be checked against a waveform whose parameters we
chose, which is the only way to tell a working estimator from one that merely
produces plausible-looking numbers.

Each generator produces the **core** waveform; `pad` frames it in noise with
cosine ramps at the edges, so the burst detector has something realistic to
find and the spectrum has no splatter from a hard on/off transition.

The GFSK generator is worth reading closely. It builds an NRZ symbol stream,
shapes it with a Gaussian pulse, then integrates to phase — frequency
modulation is *integration* of the shaped symbol stream, which is why
`np.cumsum` appears where a naive implementation would put a multiply.

In [ ]:
def awgn(x, snr_db):
    p = np.mean(np.abs(x) ** 2)
    n = np.sqrt(p / (2 * 10 ** (snr_db / 10)))
    return x + n * (RNG.standard_normal(x.size) + 1j * RNG.standard_normal(x.size))


def pad(x, fs, lead=200e-6, tail=200e-6, snr_db=25):
    """Frame a burst in noise so the burst detector has something to find."""
    nl, nt = int(lead * fs), int(tail * fs)
    p = np.mean(np.abs(x) ** 2)
    nfloor = np.sqrt(p / (2 * 10 ** (snr_db / 10)))
    noise = lambda n: nfloor * (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
    # cosine ramps to avoid spectral splatter at the edges
    r = int(min(64, x.size // 20)) or 1
    w = np.ones(x.size)
    w[:r] = np.sin(np.linspace(0, np.pi / 2, r)) ** 2
    w[-r:] = np.cos(np.linspace(0, np.pi / 2, r)) ** 2
    return np.concatenate([noise(nl), awgn(x * w, snr_db), noise(nt)]).astype(np.complex64)

In [ ]:
def gen_gfsk(fs, rate, dev, nbits, bt=0.5, m=2):
    sps = int(round(fs / rate))
    syms = RNG.integers(0, m, nbits)
    levels = (2 * syms - (m - 1)) / (m - 1)          # -1..+1
    up = np.repeat(levels, sps)
    # Gaussian pulse shaping
    span = 4 * sps
    t = (np.arange(-span, span + 1)) / sps
    a = np.sqrt(np.log(2) / 2) / bt
    g = np.exp(-(np.pi**2) * (t**2) / (2 * a**2))
    g /= g.sum()
    up = np.convolve(up, g, mode="same")
    ph = 2 * np.pi * dev * np.cumsum(up) / fs
    return np.exp(1j * ph).astype(np.complex64)


def gen_ofdm(fs, n_fft, n_cp, n_active, n_sym):
    out = []
    idx = np.r_[np.arange(1, n_active // 2 + 1), np.arange(n_fft - n_active // 2, n_fft)]
    for _ in range(n_sym):
        X = np.zeros(n_fft, complex)
        X[idx] = (RNG.choice([-1, 1], idx.size) + 1j * RNG.choice([-1, 1], idx.size)) / np.sqrt(2)
        s = np.fft.ifft(X) * np.sqrt(n_fft)
        out.append(np.r_[s[-n_cp:], s])
    return np.concatenate(out).astype(np.complex64)


def gen_lora(fs, bw, sf, n_sym):
    n = int(round(2**sf * fs / bw))
    k = bw / (2**sf / bw)                            # Hz/s
    t = np.arange(n) / fs
    out = []
    for _ in range(n_sym):
        off = RNG.integers(0, 2**sf) / 2**sf
        f = (-bw / 2 + bw * ((t / (n / fs) + off) % 1.0))
        out.append(np.exp(1j * 2 * np.pi * np.cumsum(f) / fs))
    return np.concatenate(out).astype(np.complex64)


def gen_ppm_ook(fs, chip_rate, nchips):
    """ADS-B-style pulse-position modulation: 0.5 us pulse in each 1 us chip."""
    sps = int(round(fs / chip_rate))
    half = sps // 2
    out = np.zeros(nchips * sps)
    for i in range(nchips):
        o = 0 if RNG.integers(0, 2) else half
        out[i * sps + o : i * sps + o + half] = 1.0
    out = sig.lfilter(np.ones(max(2, sps // 8)) / max(2, sps // 8), 1, out)
    return out.astype(np.complex64)


def gen_linear(fs, rate, nsym, order=4, alpha=0.35):
    sps = int(round(fs / rate))
    ang = 2 * np.pi * RNG.integers(0, order, nsym) / order
    syms = np.exp(1j * ang)
    up = np.zeros(nsym * sps, complex)
    up[::sps] = syms
    span = 8
    t = np.arange(-span * sps, span * sps + 1) / sps
    with np.errstate(divide="ignore", invalid="ignore"):
        h = np.sinc(t) * np.cos(np.pi * alpha * t) / (1 - (2 * alpha * t) ** 2)
    h[~np.isfinite(h)] = 0
    h[np.abs(1 - (2 * alpha * t) ** 2) < 1e-9] = alpha / 2 * np.sin(np.pi / (2 * alpha))
    h /= np.sqrt(np.sum(h**2))
    return np.convolve(up, h, mode="same").astype(np.complex64)

A quick look at what these produce. Note how visually distinct the four
modulation families already are in the time-frequency plane — that separability
is what the blind features are trying to capture numerically.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(13, 2.9))

demos = [
    ("GFSK 250 kbps\n(SiK class)", gen_gfsk(2e6, 250e3, 60e3, 120), 2e6),
    ("OFDM N=64 CP=16\n(Wi-Fi class)", gen_ofdm(20e6, 64, 16, 52, 40), 20e6),
    ("LoRa BW125 SF7\n(CSS)", gen_lora(500e3, 125e3, 7, 8), 500e3),
    ("PPM/OOK\n(ADS-B class)", gen_ppm_ook(8e6, 1e6, 40), 8e6),
]
for a, (title, x, fs) in zip(ax, demos):
    nper = max(32, min(256, len(x) // 24))
    a.specgram(x, NFFT=nper, Fs=fs / 1e6, noverlap=nper // 2, cmap="magma")
    a.set_title(title, fontsize=8)
    a.set_xlabel("time (us)" if fs > 1e6 else "time (s)")
    a.grid(False)
ax[0].set_ylabel("MHz")
plt.tight_layout()
plt.show()

## 4. Reading captures from disk

Nothing subtle, but two details cause most real-world grief. **Interleaving:**
integer formats store $I_0 Q_0 I_1 Q_1 \dots$, so an odd sample count means a
truncated file. **RTL-SDR's `cu8`** is unsigned with a 127.5 offset; forgetting
to remove it puts a large DC spike at bin zero that corrupts every bandwidth
measurement downstream.

Normalising to unit RMS makes every threshold in the rest of the notebook
scale-free, which matters because SDR sample scaling is arbitrary and varies
with gain settings.

In [ ]:
_DTYPES = {
    "cf32": (np.complex64, False),
    "cf64": (np.complex128, False),
    "sc16": (np.int16, True),
    "sc8": (np.int8, True),
    "cu8": (np.uint8, True),   # RTL-SDR
}


def load_iq(path: str, fmt: str = "cf32", max_samples: Optional[int] = None) -> np.ndarray:
    """Read an interleaved IQ file into complex64."""
    if fmt not in _DTYPES:
        raise ValueError(f"unknown format {fmt!r}; choose from {sorted(_DTYPES)}")
    dtype, interleaved = _DTYPES[fmt]
    count = -1 if max_samples is None else (max_samples * 2 if interleaved else max_samples)
    raw = np.fromfile(path, dtype=dtype, count=count)
    if interleaved:
        if raw.size % 2:
            raw = raw[:-1]
        raw = raw.astype(np.float32)
        if dtype is np.uint8:
            raw -= 127.5
        x = raw[0::2] + 1j * raw[1::2]
    else:
        x = raw
    x = np.asarray(x, dtype=np.complex64)
    # normalise to unit RMS so thresholds are scale-free
    rms = np.sqrt(np.mean(np.abs(x) ** 2)) or 1.0
    return (x / rms).astype(np.complex64)

# Part II — Blind feature extraction

## 5. Burst segmentation

Most captures are mostly silence. Before measuring anything we split the record
into bursts, because averaging a 200 µs frame together with 5 ms of noise
destroys every statistic we care about.

### Energy detection

Smooth the instantaneous power over a short window and compare against the
noise floor:

$$P[n] = \frac{1}{W}\sum_{k=0}^{W-1}\bigl|x[n-k]\bigr|^2, \qquad
\text{declare signal where } 10\log_{10} P[n] > \eta_{\text{noise}} + \tau.$$

We use **hysteresis** — a high threshold (8 dB) to open a burst and a lower one
(5 dB) to close it. A single threshold chatters on and off through the burst's
own amplitude fluctuations, fragmenting one frame into dozens of pieces.

### Estimating the noise floor is the hard part

The obvious choice, a low percentile of $P$, fails whenever the duty cycle is
high: in a capture that is 85% signal, the 20th percentile lies *inside the
signal*, so the estimated floor is the signal level, the threshold is never
exceeded, and nothing is detected. This is not hypothetical — it was a real bug,
and it silently broke the LoRa and DMR test cases while the code appeared to
work.

The fix is the **minimum of per-block medians**. Split the record into ~200
blocks and take a low percentile of the block medians. This needs only *one*
quiet window anywhere in the record, regardless of overall duty cycle. It also
degrades gracefully: for a genuinely continuous carrier every block median sits
at the signal level, the peak-to-floor difference collapses, and that condition
is exactly what triggers the continuous-signal fallback.

In [ ]:
def _smooth(x: np.ndarray, n: int) -> np.ndarray:
    if n <= 1:
        return x
    k = np.ones(n, dtype=np.float64) / n
    return np.convolve(x, k, mode="same")


def _noise_floor_db(p_db: np.ndarray, n_blocks: int = 200) -> float:
    """Noise floor = the quietest block median.

    A global low percentile fails whenever the duty cycle is high (an 85%-on
    capture puts the 20th percentile inside the signal). Taking the minimum of
    per-block medians only needs *one* quiet window anywhere in the record, and
    degrades gracefully to the signal level for a truly continuous carrier —
    which is exactly what makes the continuous fallback fire.
    """
    n = p_db.size
    blk = max(16, n // n_blocks)
    m = (n // blk) * blk
    if m < blk:
        return float(np.percentile(p_db, 10.0))
    med = np.median(p_db[:m].reshape(-1, blk), axis=1)
    return float(np.percentile(med, 2.0)) if med.size >= 20 else float(med.min())

In [ ]:
@dataclass
class Burst:
    i0: int
    i1: int
    fs: float

    @property
    def duration(self) -> float:
        return (self.i1 - self.i0) / self.fs

    @property
    def t0(self) -> float:
        return self.i0 / self.fs


def detect_bursts(
    x: np.ndarray,
    fs: float,
    win_s: float = 2e-6,
    on_db: float = 8.0,
    off_db: float = 5.0,
    min_dur_s: float = 4e-6,
    max_gap_s: float = 8e-6,
    guard_frac: float = 0.05,
) -> tuple[list[Burst], float]:
    """Energy-based burst detector with hysteresis.

    Returns (bursts, noise_floor_db). If the signal turns out to be
    continuous, a single burst spanning the whole record is returned.
    """
    # Floors in *samples*, not just seconds. These time constants were chosen
    # for MHz-rate captures; at 48 ksps a 2 us window is 0.1 samples, leaving the
    # power envelope unsmoothed and gap-merging disabled, which fragments one
    # burst into pieces. The largest fragment then has too few symbols for the
    # eye histogram in fsk_tones, and narrowband 4-FSK silently degrades to
    # 2 tones. Floors keep the detector meaningful at any sample rate.
    n_win = max(4, int(round(win_s * fs)))
    p = _smooth(np.abs(x).astype(np.float64) ** 2, n_win)
    p_db = 10.0 * np.log10(p + 1e-20)

    noise_db = _noise_floor_db(p_db)
    peak_db = float(np.percentile(p_db, 99.9))
    if peak_db - noise_db < 3.0:
        # no discernible on/off structure -> treat as continuous
        return [Burst(0, len(x), fs)], noise_db

    hi = p_db > (noise_db + on_db)
    lo = p_db > (noise_db + off_db)

    # hysteresis: grow every hi region outward while lo holds
    state = np.zeros(len(p_db), dtype=bool)
    idx = np.flatnonzero(hi)
    if idx.size == 0:
        return [Burst(0, len(x), fs)], noise_db
    edges = np.flatnonzero(np.diff(idx) > 1)
    starts = np.r_[idx[0], idx[edges + 1]]
    stops = np.r_[idx[edges], idx[-1]]
    for a, b in zip(starts, stops):
        while a > 0 and lo[a - 1]:
            a -= 1
        while b < len(lo) - 1 and lo[b + 1]:
            b += 1
        state[a : b + 1] = True

    # runs -> bursts, merging short gaps
    d = np.diff(state.astype(np.int8))
    ons = list(np.flatnonzero(d == 1) + 1)
    offs = list(np.flatnonzero(d == -1) + 1)
    if state[0]:
        ons.insert(0, 0)
    if state[-1]:
        offs.append(len(state))

    max_gap = max(8, int(round(max_gap_s * fs)))
    merged: list[list[int]] = []
    for a, b in zip(ons, offs):
        if merged and a - merged[-1][1] <= max_gap:
            merged[-1][1] = b
        else:
            merged.append([a, b])

    min_len = max(16, int(round(min_dur_s * fs)))
    bursts = []
    for a, b in merged:
        if b - a < min_len:
            continue
        g = int((b - a) * guard_frac)  # trim ramps
        bursts.append(Burst(a + g, b - g, fs))
    if not bursts:
        return [Burst(0, len(x), fs)], noise_db

    # many fragments covering nearly everything => continuous carrier that the
    # hysteresis chopped up; a single burst with quiet margins is left alone.
    covered = sum(b.i1 - b.i0 for b in bursts)
    if len(bursts) > 2 and covered > 0.85 * len(x):
        return [Burst(0, len(x), fs)], noise_db
    return bursts, noise_db

Demonstrating both the failure and the fix on a high-duty-cycle record.

In [ ]:
fs = 500e3
core = gen_lora(fs, 125e3, 7, 40)
iq = pad(core, fs, 4e-3, 4e-3)          # ~84% duty cycle
p_db = 10 * np.log10(_smooth(np.abs(iq) ** 2, 4) + 1e-20)

naive = np.percentile(p_db, 20.0)
robust = _noise_floor_db(p_db)
true_floor = 10 * np.log10(np.mean(np.abs(iq[:1500]) ** 2))

t = np.arange(iq.size) / fs * 1e3
plt.plot(t, p_db, lw=0.4, color="0.6", label="smoothed power")
for v, lab, c in ((naive, "20th percentile", "tab:red"),
                  (robust, "min block median", "tab:green"),
                  (true_floor, "true noise floor", "k")):
    plt.axhline(v, color=c, ls="--", lw=1.2, label=f"{lab}: {v:.1f} dB")
plt.xlabel("time (ms)"); plt.ylabel("dB"); plt.legend(fontsize=7, ncol=2)
plt.title("Noise-floor estimation at 84% duty cycle")
plt.show()

print(f"20th percentile overestimates the floor by {naive - true_floor:5.1f} dB")
print(f"min block median is off by         {robust - true_floor:5.1f} dB")
b, nd = detect_bursts(iq, fs)
print(f"\nbursts found: {len(b)}, spanning "
      f"{[(round(x.t0*1e3,2), round(x.duration*1e3,2)) for x in b]} (ms)")

## 6. Spectral estimation and bandwidth

### Welch's method

A single periodogram of an $M$-sample record has variance equal to the square of
its own mean — it is a consistent estimator of nothing. Welch averages $K$
overlapping windowed periodograms, trading frequency resolution for variance
reduction by roughly $1/K$:

$$\hat{S}(f) = \frac{1}{K}\sum_{i=1}^{K}
\frac{1}{\|w\|^2}\Bigl|\sum_{n} x_i[n]\,w[n]\,e^{-j2\pi fn/f_s}\Bigr|^2$$

We keep the two-sided spectrum, because a baseband burst sitting off-centre in
the capture has a frequency offset we need to measure and remove.

### Occupied bandwidth

The standard regulatory definition: the band containing a fraction $\alpha$
(here 99%) of total power, split symmetrically about the tails. With the
cumulative distribution $C(f) = \int_{-f_s/2}^{f} \hat S / \int \hat S$,

$$B_{99} = C^{-1}\!\left(1 - \tfrac{1-\alpha}{2}\right) - C^{-1}\!\left(\tfrac{1-\alpha}{2}\right),
\qquad f_{\text{centre}} = \tfrac{1}{2}\bigl(\text{upper} + \text{lower}\bigr).$$

Interpolating $C^{-1}$ rather than snapping to bins matters: at coarse
resolution, bin snapping quantises the bandwidth badly enough to push a
narrowband signal into the wrong protocol's range.

### RMS bandwidth

The second-moment bandwidth $B_{\text{rms}} = 2\sqrt{\int (f-\mu)^2 \hat S / \int \hat S}$
weights energy far from the centre quadratically. That makes it a useful
companion to $B_{99}$ — but only if it is computed **inside the occupied band**.
Integrating over the whole capture folds the out-of-band noise floor into the
second moment; before this was restricted, a 320 kHz burst in a 20 MHz capture
reported an RMS bandwidth of 1.14 MHz, over three times too large.

### Shape features

**Spectral flatness** (Wiener entropy) is the ratio of geometric to arithmetic
mean of the PSD inside the band:

$$\mathcal{F} = \frac{\exp\bigl(\frac{1}{K}\sum_k \ln S_k\bigr)}{\frac{1}{K}\sum_k S_k} \in (0, 1]$$

It approaches 1 for OFDM, whose many independent subcarriers fill the band
uniformly, and drops for the rounded spectrum of a shaped single carrier.

**Edge sharpness** measures dB of roll-off just outside the band edge,
separating brick-wall filtered signals from slowly decaying ones.

In [ ]:
def welch_psd(x: np.ndarray, fs: float, nperseg: Optional[int] = None):
    n = len(x)
    if nperseg is None:
        nperseg = int(2 ** np.floor(np.log2(max(64, min(4096, n // 4 or 64)))))
    nperseg = min(nperseg, n)
    f, pxx = signal.welch(
        x, fs=fs, nperseg=nperseg, noverlap=nperseg // 2,
        return_onesided=False, detrend=False, scaling="density",
    )
    order = np.argsort(f)
    return f[order], pxx[order]


def occupied_bandwidth(f: np.ndarray, pxx: np.ndarray, frac: float = 0.99):
    """Return (bandwidth, center) containing `frac` of total power."""
    total = np.sum(pxx)
    if total <= 0:
        return 0.0, 0.0
    c = np.cumsum(pxx) / total
    tail = (1.0 - frac) / 2.0
    lo = float(np.interp(tail, c, f))
    hi = float(np.interp(1.0 - tail, c, f))
    return hi - lo, 0.5 * (lo + hi)


def rms_bandwidth(f: np.ndarray, pxx: np.ndarray,
                  bw: Optional[float] = None, center: float = 0.0) -> float:
    """RMS (2nd-moment) bandwidth, restricted to the occupied band.

    Integrating over the whole capture would fold the out-of-band noise floor
    into the second moment and report a value several times too large for a
    narrowband burst in a wide capture.
    """
    if bw:
        m = np.abs(f - center) <= bw
        if m.sum() >= 8:
            f, pxx = f[m], pxx[m]
    w = pxx / (np.sum(pxx) + 1e-30)
    mu = float(np.sum(w * f))
    return 2.0 * math.sqrt(max(0.0, float(np.sum(w * (f - mu) ** 2))))


def spectral_flatness(pxx: np.ndarray, f: np.ndarray, bw: float, center: float) -> float:
    """Geometric/arithmetic mean ratio inside the occupied band.

    ~1.0 for OFDM (flat), lower for shaped single-carrier / FSK humps.
    """
    m = np.abs(f - center) <= (bw / 2.0)
    if m.sum() < 8:
        return 0.0
    p = pxx[m] + 1e-30
    return float(np.exp(np.mean(np.log(p))) / np.mean(p))


def edge_sharpness(pxx: np.ndarray, f: np.ndarray, bw: float, center: float) -> float:
    """dB of roll-off per 10% of BW just outside the band edge.

    High for brick-wall (OFDM, filtered), low for slowly decaying spectra.
    """
    step = bw * 0.10
    def band_db(lo, hi):
        m = (f >= lo) & (f <= hi)
        return 10 * np.log10(np.mean(pxx[m]) + 1e-30) if m.sum() else np.nan
    inner = band_db(center - bw * 0.3, center + bw * 0.3)
    outer = band_db(center + bw / 2 + 0.02 * bw, center + bw / 2 + step)
    outer2 = band_db(center - bw / 2 - step, center - bw / 2 - 0.02 * bw)
    vals = [v for v in (inner - outer, inner - outer2) if np.isfinite(v)]
    return float(np.mean(vals)) if vals else 0.0

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.1))
cases = [("GFSK 250 kbps", gen_gfsk(2e6, 250e3, 60e3, 400), 2e6),
         ("OFDM 52/64 carriers", gen_ofdm(20e6, 64, 16, 52, 200), 20e6),
         ("QPSK RRC a=0.35", gen_linear(20e6, 5e6, 800), 20e6)]

for a, (title, x, fs) in zip(ax, cases):
    f, pxx = welch_psd(x, fs)
    bw, c = occupied_bandwidth(f, pxx, 0.99)
    a.plot(f / 1e6, 10 * np.log10(pxx / pxx.max() + 1e-20), lw=0.7)
    a.axvspan((c - bw / 2) / 1e6, (c + bw / 2) / 1e6, alpha=0.15, color="tab:green")
    a.set_title(f"{title}\nB99={bw/1e3:.0f} kHz  "
                f"flatness={spectral_flatness(pxx, f, bw, c):.2f}  "
                f"edge={edge_sharpness(pxx, f, bw, c):.0f} dB", fontsize=8)
    a.set_xlabel("MHz"); a.set_ylim(-60, 3)
ax[0].set_ylabel("dB (normalised)")
plt.tight_layout(); plt.show()

## 7. Amplitude statistics, and two numbers we can predict exactly

Two cheap scalars separate constant-envelope modulations (FSK, MSK, GMSK, CSS)
from those carrying information in amplitude (OFDM, QAM, OOK).

### Envelope coefficient of variation

$$\mathrm{CV} = \frac{\operatorname{std}|x|}{\operatorname{mean}|x|}$$

For an ideal constant-envelope signal, $\mathrm{CV} \to 0$. For OFDM we can
derive the value in closed form. With many independent subcarriers the central
limit theorem makes each time-domain sample circularly complex Gaussian, so
$|x|$ is **Rayleigh** with parameter $\sigma$:

$$\mathbb{E}|x| = \sigma\sqrt{\tfrac{\pi}{2}}, \qquad \mathbb{E}|x|^2 = 2\sigma^2
\;\Rightarrow\; \operatorname{Var}|x| = \sigma^2\!\left(2 - \tfrac{\pi}{2}\right)$$

$$\boxed{\;\mathrm{CV}_{\text{Rayleigh}} = \frac{\sigma\sqrt{2 - \pi/2}}{\sigma\sqrt{\pi/2}}
= \sqrt{\frac{4}{\pi} - 1} \approx 0.5227\;}$$

This is a genuinely sharp prediction with no free parameters, and it is the
single most reliable OFDM indicator in the whole feature set.

### Peak-to-average power ratio

For $M$ i.i.d. exponential power samples, the expected maximum grows
logarithmically:

$$\mathbb{E}\!\left[\frac{\max_m |x_m|^2}{\mathbb{E}|x|^2}\right] \approx \ln M + \gamma_E,
\qquad \gamma_E \approx 0.5772$$

so PAPR in dB is $\approx 10\log_{10}(\ln M + 0.577)$ — slowly growing, and
therefore weak evidence on its own. It is genuinely useful only for pulsed
signals, where a 50%-duty OOK waveform gives almost exactly 3 dB.

Let us check both predictions.

In [ ]:
def env_stats(x):
    a = np.abs(x).astype(float)
    cv = a.std() / a.mean()
    papr = 10 * np.log10(a.max() ** 2 / np.mean(a ** 2))
    return cv, papr

rows = [
    ("GFSK (constant envelope)", gen_gfsk(2e6, 250e3, 60e3, 800), 0.0),
    ("OFDM 52/64 carriers", gen_ofdm(20e6, 64, 16, 52, 400), math.sqrt(4/math.pi - 1)),
    ("QPSK RRC a=0.35", gen_linear(20e6, 5e6, 2000), None),
    ("PPM/OOK 50% duty", gen_ppm_ook(8e6, 1e6, 400), 1.0),
]
print(f"{'waveform':30s} {'CV':>8} {'predicted':>10} {'PAPR dB':>9} {'ln M+g':>8}")
for name, x, pred in rows:
    cv, papr = env_stats(x)
    p = f"{pred:.4f}" if pred is not None else "—"
    print(f"{name:30s} {cv:8.4f} {p:>10} {papr:9.2f} "
          f"{10*math.log10(math.log(len(x)) + 0.5772):8.2f}")

print(f"\nRayleigh CV prediction: sqrt(4/pi - 1) = {math.sqrt(4/math.pi - 1):.4f}")
print("The OFDM row matches to ~0.01 with no fitted parameters.")

## 8. Channel filtering and decimation

A 250 kHz burst in a 20 MHz capture is 80 samples per symbol of mostly noise.
Filtering to the occupied band and decimating discards the out-of-band noise —
the largest cheap SNR win available, worth roughly $10\log_{10}(f_s/2B)$, about
9 dB in this example.

It also makes sample-count-based smoothing windows *meaningful*: a window of
"5 samples" means something entirely different at 80 samples/symbol than at 6.

The target rate is a compromise with a real cost on each side. Too little
decimation and the discriminator smoothing in §12 cannot be tuned; too much and
we starve the eye of samples per symbol. Targeting $\approx 6B$ works; targeting
$4B$ **broke the BLE test case**, because at 4 samples/symbol a GFSK eye stops
resolving reliably. That is why the guard is `decim >= 3` rather than `>= 2`.

In [ ]:
fs, bw = 20e6, 320e3
for target_mult in (4.0, 6.0, 10.0):
    d = int(max(1, np.floor(fs / (target_mult * bw))))
    print(f"target {target_mult:4.1f}xB -> decimate {d:3d}x  "
          f"-> fs={fs/d/1e6:6.3f} MHz, {fs/d/250e3:5.2f} samples/symbol at 250 kbps, "
          f"noise rejected ~{10*math.log10(max(d,1)):4.1f} dB")

## 9. The FM discriminator and its noise threshold

Almost every narrowband feature we extract comes from the **instantaneous
frequency**, obtained by differencing unwrapped phase:

$$\hat f[n] = \frac{f_s}{2\pi}\Bigl(\phi[n+1] - \phi[n]\Bigr)$$

This is the single noisiest operation in the pipeline, and understanding exactly
*how* noisy is what makes the rest of Part II work.

### Phase noise to frequency noise

Write the received sample as signal plus noise, $x = A e^{j\phi} + w$. For small
noise, the phase error is the noise component in quadrature with the signal,
divided by the amplitude:

$$\theta \approx \frac{\operatorname{Im}\{w e^{-j\phi}\}}{A}
\qquad\Longrightarrow\qquad
\sigma_\theta^2 = \frac{\sigma^2/2}{A^2} = \frac{1}{2\gamma}$$

Differencing two *independent* phase errors doubles the variance, so

$$\sigma_{\hat f}^2 = 2\sigma_\theta^2\left(\frac{f_s}{2\pi}\right)^2
= \frac{1}{\gamma}\left(\frac{f_s}{2\pi}\right)^2
\qquad\Longrightarrow\qquad
\boxed{\;\sigma_{\hat f} = \frac{f_s}{2\pi\sqrt{\gamma}}\;}$$

Put numbers in: at $f_s = 48$ kHz and 25 dB SNR, $\sigma_{\hat f} \approx 430$ Hz.
A DMR 4-FSK signal has adjacent tones **1296 Hz** apart. The noise is a third of
the tone spacing, which is why raw discriminator histograms of narrowband 4-FSK
collapse into a single blob.

### Smoothing telescopes

Averaging $W$ consecutive difference estimates does something better than
$\sqrt{W}$, because the sum telescopes:

$$\frac{1}{W}\sum_{k=0}^{W-1}\bigl(\phi[n{+}k{+}1]-\phi[n{+}k]\bigr)
= \frac{\phi[n{+}W]-\phi[n]}{W}$$

Only the two endpoint phase errors survive, so

$$\sigma_{\hat f, W} = \frac{f_s}{2\pi W \sqrt{\gamma}} \quad\text{— falls as } 1/W,\ \text{not } 1/\sqrt{W}.$$

### The constraint that closes the argument

Smoothing is only valid while the frequency is constant, so $W$ cannot exceed
the symbol period, $W < f_s/R_s$. Requiring $\pm 3\sigma$ separation between
adjacent tones, $\sigma_{\hat f,W} \le \Delta f/6$, gives

$$W \;\ge\; \frac{3 f_s}{\pi \sqrt{\gamma}\, \Delta f}
\qquad\text{and combining with } W < f_s/R_s:$$

$$\boxed{\;\gamma \;>\; \left(\frac{3 R_s}{\pi \Delta f}\right)^{2}\;}$$

A hard SNR floor for tone separation that depends only on the ratio of symbol
rate to tone spacing. For DMR ($R_s = 4800$, $\Delta f = 1296$) it is **11 dB**;
for BLE ($R_s = 10^6$, $\Delta f = 500$ kHz) it is **5.6 dB**. Narrowband M-ary
FSK is intrinsically the hardest case, and no estimator design escapes this.

In [ ]:
def inst_freq(x: np.ndarray, fs: float, mag_gate: float = 0.35) -> np.ndarray:
    """Instantaneous frequency (Hz), gated to samples above `mag_gate` * median |x|."""
    ph = np.unwrap(np.angle(x.astype(np.complex128)))
    f = np.diff(ph) * fs / (2 * np.pi)
    a = np.abs(x[1:])
    gate = mag_gate * np.median(np.abs(x)) if np.median(np.abs(x)) > 0 else 0.0
    return f[a > gate]

In [ ]:
fs, snr_db = 48e3, 25.0
clean = gen_gfsk(fs, 4800, 1944, 400, bt=0.6, m=4)
noisy = awgn(clean, snr_db)

gamma = 10 ** (snr_db / 10)
pred = fs / (2 * np.pi * math.sqrt(gamma))
meas = np.std(inst_freq(noisy, fs) - inst_freq(clean, fs)[:inst_freq(noisy, fs).size])

print(f"predicted sigma_f = fs/(2*pi*sqrt(gamma)) = {pred:7.1f} Hz")
print(f"measured                                  = {meas:7.1f} Hz")
print(f"\nDMR 4-FSK adjacent tone spacing           = {2*1944/3:7.1f} Hz")

print("\nTone-separation SNR floor, gamma > (3*Rs/(pi*df))^2:")
for nm, rs, df in (("DMR 4-FSK", 4800, 2*1944/3), ("BLE 1M 2-FSK", 1e6, 500e3),
                   ("SiK 2-FSK", 250e3, 120e3), ("POCSAG 2-FSK", 1200, 9000)):
    g = (3 * rs / (math.pi * df)) ** 2
    print(f"   {nm:14s} Rs={rs:8.0f}  df={df:8.0f} -> {10*math.log10(g):+6.1f} dB")

W = np.arange(1, 33)
plt.semilogy(W, fs / (2 * np.pi * W * math.sqrt(gamma)), label=r"telescoping: $\propto 1/W$")
plt.semilogy(W, fs / (2 * np.pi * np.sqrt(W) * math.sqrt(gamma)), ls="--",
             label=r"naive averaging: $\propto 1/\sqrt{W}$")
plt.axhline(2 * 1944 / 3 / 6, color="k", ls=":", label=r"required: $\Delta f/6$")
plt.axvline(fs / 4800, color="tab:red", ls=":", label="one symbol (upper limit on W)")
plt.xlabel("smoothing window W (samples)"); plt.ylabel("$\\sigma_f$ (Hz)")
plt.legend(fontsize=7); plt.title("Discriminator noise vs smoothing, DMR at 25 dB")
plt.show()

## 10. OFDM: detecting the cyclic prefix

An OFDM symbol is an IFFT of $N$ subcarriers, prefixed by a copy of its own
last $L$ samples:

$$x[n] = x[n+N] \qquad \text{for the } L \text{ samples of each cyclic prefix.}$$

That exact repetition is a strong, protocol-agnostic signature. Correlate the
signal against a delayed copy of itself:

$$\hat R(\tau) = \sum_n x[n]\,x^*[n+\tau]$$

computed via FFT in $O(M\log M)$. At $\tau = N$ the CP samples add
**coherently** while everything else adds incoherently, so with $M$ total
samples:

$$\frac{|\hat R(N)|}{\hat R(0)} \;\approx\; \frac{L}{N+L},
\qquad \text{noise floor of the estimate} \;\sim\; \frac{1}{\sqrt{M}}$$

Requiring a detection SNR of about 5 gives the record-length condition

$$\boxed{\;M \;\gtrsim\; 25\left(\frac{N+L}{L}\right)^{2}\;}$$

For Wi-Fi ($N{=}64$, $L{=}16$) that is only 625 samples. For LTE ($N{=}1024$,
$L{=}72$) it is 5800. **This is why the threshold matters:** LTE's normal cyclic
prefix is 6.6% of the symbol, so a hard threshold of 0.08 on the normalised peak
rejects LTE outright — which it did, until the threshold was dropped to 0.03 and
the decision moved onto the robust $z$-score instead.

Recovering $N$ gives the **subcarrier spacing** $\Delta f_{sc} = f_s/N$
directly, and that single number separates Wi-Fi (312.5 kHz) from LTE (15 kHz)
from 5G NR (30 kHz) from DAB (1 kHz) — the most decisive measurement in the
whole classifier.

Two guards worth noting. A peak pinned to either end of the search range is
rejected: a chirp has high short-lag autocorrelation and will otherwise
masquerade as OFDM with a tiny FFT size. And repeated training symbols (802.11's
L-STF is ten repeats of 16 samples) also produce peaks here — harmless, since
they still indicate OFDM, but it means the recovered "FFT size" is occasionally
a training-sequence period.

In [ ]:
def cp_autocorrelation(x: np.ndarray, min_lag: int, max_lag: int):
    """Normalised |R(lag)| via FFT. OFDM cyclic prefixes create an isolated peak
    at lag == FFT size; repeated training symbols also show up here."""
    n = len(x)
    max_lag = int(min(max_lag, n // 3))
    if max_lag <= min_lag:
        return None
    nfft = 1 << int(np.ceil(np.log2(2 * n)))
    X = np.fft.fft(x.astype(np.complex128), nfft)
    r = np.fft.ifft(X * np.conj(X))[: max_lag + 1]
    r = np.abs(r) / (abs(r[0]) + 1e-30)
    lags = np.arange(min_lag, max_lag + 1)
    vals = r[min_lag : max_lag + 1]
    if vals.size < 8:
        return None
    k = int(np.argmax(vals))
    if k == 0 or k == vals.size - 1:
        return None                      # peak pinned to the search boundary
    med = float(np.median(vals))
    mad = float(np.median(np.abs(vals - med))) + 1e-12
    return {
        "lag": int(lags[k]),
        "value": float(vals[k]),
        "score": float((vals[k] - med) / mad),   # robust z-score
        "curve": (lags, vals),
    }


def estimate_cp_length(x: np.ndarray, n_fft: int) -> Optional[int]:
    """Given a candidate FFT size, find the CP length that maximises the
    sliding-window correlation periodicity."""
    best, best_cp = -1.0, None
    for cp in (n_fft // 32, n_fft // 16, n_fft // 8, n_fft // 4):
        if cp < 2 or len(x) < 3 * (n_fft + cp):
            continue
        prod = x[:-n_fft] * np.conj(x[n_fft:])
        w = np.ones(cp) / cp
        g = np.abs(np.convolve(prod, w, mode="valid"))
        if g.size < 4:
            continue
        contrast = float(np.percentile(g, 98) / (np.median(g) + 1e-30))
        if contrast > best:
            best, best_cp = contrast, cp
    return best_cp

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for a, (title, x, fs, Nt, Lt) in zip(ax, [
        ("Wi-Fi class: N=64, CP=16", gen_ofdm(20e6, 64, 16, 52, 400), 20e6, 64, 16),
        ("LTE class: N=1024, CP=72", gen_ofdm(15.36e6, 1024, 72, 600, 60), 15.36e6, 1024, 72)]):
    x = awgn(x, 25)
    ac = cp_autocorrelation(x, min_lag=8, max_lag=2048)
    lags, vals = ac["curve"]
    a.plot(lags, vals, lw=0.6)
    a.axvline(Nt, color="tab:red", ls="--", lw=1, label=f"true N={Nt}")
    a.axhline(Lt / (Nt + Lt), color="tab:green", ls=":", label=f"L/(N+L)={Lt/(Nt+Lt):.3f}")
    cp = estimate_cp_length(x, ac["lag"])
    a.set_title(f"{title}\nfound N={ac['lag']}, CP={cp}, z={ac['score']:.0f}, "
                f"SCS={fs/ac['lag']/1e3:.1f} kHz", fontsize=8)
    a.set_xlabel("lag (samples)"); a.legend(fontsize=7)
    print(f"{title}: predicted peak {Lt/(Nt+Lt):.4f}, measured {ac['value']:.4f}, "
          f"min record 25((N+L)/L)^2 = {25*((Nt+Lt)/Lt)**2:.0f} samples")
ax[0].set_ylabel(r"$|\hat R(\tau)|/\hat R(0)$")
plt.tight_layout(); plt.show()

## 11. Chirp spread spectrum (LoRa): detect by dechirping

A LoRa symbol sweeps the entire bandwidth $B$ exactly once per symbol. With
spreading factor $\mathrm{SF}$, the symbol period and chirp rate are

$$T_s = \frac{2^{\mathrm{SF}}}{B}, \qquad
k = \frac{B}{T_s} = \frac{B^2}{2^{\mathrm{SF}}}\ \ [\text{Hz/s}],
\qquad f(t) = -\frac{B}{2} + kt \ (\mathrm{mod}\ B)$$

### The approach that failed

The natural idea is to test whether $\mathrm{d}f/\mathrm{d}t$ is constant. It
does not work, and §9 tells us exactly why. Differentiating the discriminator a
*second* time compounds the noise: at 25 dB SNR the noise on the estimated slope
came out the same order as the slope itself, and the linearity score for a true
LoRa signal was **0.27** where it needed to exceed 0.8.

### Dechirping

Multiply by the conjugate of a reference chirp. The quadratic phase cancels and
each symbol collapses to a **single tone** whose frequency is the symbol value:

$$y(t) = x(t)\,e^{-j\pi k t^2} \;\longrightarrow\; \text{tone at } f_0 = \text{symbol offset}$$

An FFT over one symbol has resolution $1/T_s = B/2^{\mathrm{SF}}$, giving exactly
$2^{\mathrm{SF}}$ resolvable bins — which is the LoRa symbol alphabet, as it must
be. We score the fraction of per-symbol FFT energy landing in the peak bin:
$\approx 0.9$ for a matched dechirp, $\approx 0.45$ if the symbol boundary is
misaligned (a cyclic chirp splits into two tones), and $\approx 1/N$ for
anything else.

### Why we search the standard bandwidth grid

A chirp-rate error $\Delta k$ leaves residual sweep, spreading the tone across

$$\Delta k \cdot T_s^2 = \frac{\Delta k}{k}\,k\,T_s^2
= \frac{\Delta k}{k}\,2^{\mathrm{SF}} \ \text{bins}
\qquad(\text{since } k T_s^2 = 2^{\mathrm{SF}})$$

Holding the spread under ~6 bins needs $\Delta k/k \le 6/2^{\mathrm{SF}}$, and
since $k \propto B^2$ we have $\Delta k/k = 2\Delta B/B$, so

$$\boxed{\;\frac{\Delta B}{B} \;\le\; \frac{3}{2^{\mathrm{SF}}}\;}$$

That is 2.3% at SF7 and **0.07%** at SF12 — far tighter than any measured $B_{99}$.
The measured bandwidth is unusable as the chirp parameter. Instead we search the
grid of standardised LoRa bandwidths, which is exact when the signal really is
LoRa. The measured $B_{99}$ (135.6 kHz for a true 125 kHz signal, 8.5% high) only
selects *which* grid entries to try.

In [ ]:
LORA_BANDWIDTHS = (7.8e3, 10.4e3, 15.6e3, 20.8e3, 31.25e3, 41.7e3,
                   62.5e3, 125e3, 250e3, 500e3)


def detect_chirp(x: np.ndarray, fs: float, bw_hint: Optional[float] = None):
    """Chirp-spread-spectrum (LoRa) detector by dechirping.

    Pointwise d(f)/dt is useless here: at 25 dB SNR the discriminator noise on
    the slope is the same order as the slope itself. Instead we multiply by a
    conjugate reference chirp for each physically plausible (BW, SF) pair and
    measure how much of the per-symbol FFT energy lands in one bin. A matched
    dechirp collapses each symbol to a tone (two tones if the symbol boundary
    is misaligned, which still concentrates far more than anything non-CSS).
    """
    if bw_hint is None or bw_hint <= 0 or x.size < 512:
        return None

    cands = [b for b in LORA_BANDWIDTHS if 0.55 * bw_hint <= b <= 1.5 * bw_hint]
    cands.append(float(bw_hint))                       # non-standard chirps

    best = None
    for bw in cands:
        for sf in range(5, 13):
            n = int(round(2 ** sf * fs / bw))           # samples per symbol
            if n < 32 or 3 * n > x.size:
                continue
            k = bw * bw / (2 ** sf)                     # chirp rate, Hz/s
            idx = np.arange(x.size)
            tt = (idx % n) / fs                         # sawtooth time base
            y = x * np.exp(-1j * np.pi * k * tt * tt)
            nblk = min(x.size // n, 48)
            if nblk < 3:
                continue
            blocks = y[: nblk * n].reshape(nblk, n)
            S = np.abs(np.fft.fft(blocks, axis=1)) ** 2
            frac = S.max(axis=1) / (S.sum(axis=1) + 1e-30)
            score = float(np.median(frac))
            if best is None or score > best["score"]:
                best = {"score": score, "bw": float(bw), "sf": int(sf),
                        "chirp_rate": float(k), "sym_len": n}
    return best

In [ ]:
fs, bw, sf = 500e3, 125e3, 7
x = awgn(gen_lora(fs, bw, sf, 24), 20)
res = detect_chirp(x, fs, bw_hint=136e3)
print(f"detected: BW={res['bw']/1e3:.1f} kHz, SF={res['sf']}, "
      f"chirp rate={res['chirp_rate']/1e6:.1f} MHz/s, energy fraction={res['score']:.3f}")
print(f"truth:    BW={bw/1e3:.1f} kHz, SF={sf}, "
      f"k=B^2/2^SF={bw**2/2**sf/1e6:.1f} MHz/s")
print(f"\nbandwidth tolerance 3/2^SF at SF{sf}: "
      f"{3/2**sf*100:.2f}%  (vs 8.5% error in the measured B99)")

n = int(round(2 ** sf * fs / bw))
k = bw * bw / 2 ** sf
tt = (np.arange(x.size) % n) / fs
y = x * np.exp(-1j * np.pi * k * tt ** 2)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.0))
ax[0].specgram(x, NFFT=128, Fs=fs / 1e3, noverlap=96, cmap="magma")
ax[0].set_title("raw LoRa: sawtooth chirps", fontsize=8); ax[0].grid(False)
ax[0].set_ylabel("kHz")
ax[1].specgram(y, NFFT=128, Fs=fs / 1e3, noverlap=96, cmap="magma")
ax[1].set_title("after matched dechirp: tones", fontsize=8); ax[1].grid(False)

for sfx in range(5, 13):
    r = detect_chirp(x, fs, bw_hint=136e3)
    pass
scores = []
sfs = list(range(5, 13))
for s in sfs:
    nn = int(round(2 ** s * fs / bw))
    kk = bw * bw / 2 ** s
    t2 = (np.arange(x.size) % nn) / fs
    yy = x * np.exp(-1j * np.pi * kk * t2 ** 2)
    nb = min(x.size // nn, 24)
    blk = yy[:nb * nn].reshape(nb, nn)
    S = np.abs(np.fft.fft(blk, axis=1)) ** 2
    scores.append(np.median(S.max(axis=1) / S.sum(axis=1)))
ax[2].stem(sfs, scores)
ax[2].axvline(sf, color="tab:red", ls="--", lw=1)
ax[2].set_xlabel("trial SF"); ax[2].set_title("peak energy fraction vs trial SF", fontsize=8)
plt.tight_layout(); plt.show()

## 12. Symbol rate from cyclostationarity

A digitally modulated signal is not wide-sense stationary — its statistics are
**periodic** with the symbol period $T$. For a linearly modulated stream
$s(t) = \sum_k a_k\,g(t-kT)$ with i.i.d. symbols,

$$\mathbb{E}\bigl|s(t)\bigr|^2 = \sigma_a^2 \sum_k \bigl|g(t-kT)\bigr|^2$$

which is periodic in $t$ with period $T$. Its Fourier series therefore contains
a **spectral line** at $1/T$, with coefficient proportional to

$$\int G(f)\,G^*\!\left(f - \tfrac{1}{T}\right)\mathrm{d}f$$

This integral is the crux. For a root-Nyquist pulse with excess bandwidth
$\alpha$, $G(f)$ and its shift by $1/T$ overlap only in the roll-off region, so
the line strength grows with $\alpha$ — and at $\alpha = 0$ (ideal Nyquist) the
line **vanishes entirely**. A perfectly shaped signal has no recoverable symbol
rate by this method. That is a real limitation, not an implementation gap.

### Three probes

Different modulations expose the periodicity in different places, so we try
three and keep the strongest line:

1. **Sign transitions of the discriminator**, $\bigl|\Delta\,\mathrm{sgn}(\hat f - \mathrm{med})\bigr|$ — an impulse train at symbol boundaries. Best for FSK/GFSK/MSK.
2. **Discriminator derivative magnitude** $|\Delta \hat f|$ — for heavily shaped FSK.
3. **Squared envelope** $|x|^2$ — for linear modulations, OOK, and pulsed signals.

### The harmonic trap

The lines appear at *every* multiple $m/T$, and the strongest is often not the
fundamental. Walking down to the fundamental needs care: an unconstrained search
over divisors let a single noise bin several octaves down hijack the estimate,
reporting a 1 Mchip/s ADS-B frame as **250 kHz**. The fix is to make the walk
**chained** — accepting $R_s/4$ requires having first accepted $R_s/2$ — and to
demand the sub-harmonic reach at least half the strength of the peak.

In [ ]:
def _spectral_lines(s: np.ndarray, fs: float, f_lo: float, f_hi: float):
    """FFT of a cyclostationary probe; return (freqs, magnitudes, noise level)."""
    n = 1 << int(np.floor(np.log2(max(s.size, 2))))
    if n < 256:
        return None
    s = s[:n].astype(np.float64)
    s = (s - s.mean()) * np.hanning(n)
    S = np.abs(np.fft.rfft(s))
    fr = np.fft.rfftfreq(n, 1.0 / fs)
    m = (fr >= max(f_lo, fs / n * 6)) & (fr <= f_hi)
    if m.sum() < 32:
        return None
    return fr[m], S[m], float(np.median(S[m])) + 1e-30


def _fundamental(fr, S, noise, snr_min: float = 3.0):
    """Strongest spectral line, walked down to its fundamental.

    The walk is *chained*: to accept Rs/4 we must first have accepted Rs/2.
    Allowing a direct jump lets a single noise bin several octaves down hijack
    the estimate, which is how a 1 Mchip/s PPM frame ends up reported as
    250 kHz.
    """
    k = int(np.argmax(S))
    f0, best_snr = float(fr[k]), float(S[k] / noise)
    if best_snr < snr_min:
        return None
    need = max(6.0, 0.5 * best_snr)
    for _ in range(4):                          # at most Rs/16
        moved = False
        for div in (2, 3):
            f_try = f0 / div
            if f_try < fr[0]:
                continue
            j = int(np.argmin(np.abs(fr - f_try)))
            w = max(1, int(0.02 * j))
            lo = max(0, j - w)
            seg = S[lo : j + w + 1]
            if seg.size and float(seg.max() / noise) > need:
                f0 = float(fr[lo + int(np.argmax(seg))])
                moved = True
                break
        if not moved:
            break
    return f0, best_snr


def estimate_symbol_rate(x: np.ndarray, fs: float, bw_hint: Optional[float] = None):
    """Spectral-line symbol-rate estimator.

    Three cyclostationary probes are tried and the strongest line wins:
      (a) sign transitions of the FM discriminator -> FSK / GFSK / MSK
      (b) magnitude of the discriminator derivative -> shaped FSK
      (c) squared envelope -> linear mods, OOK, pulsed
    Sub-harmonics are checked so a strong 2*Rs line does not double the answer.
    """
    f_lo = (bw_hint / 60.0) if bw_hint else fs / 5000.0
    f_hi = (bw_hint * 1.6) if bw_hint else fs / 2.0

    probes = []
    f = inst_freq(x, fs)
    if f.size > 512:
        d = np.sign(f - np.median(f))
        probes.append(("fm-sign-transitions", np.abs(np.diff(d))))
        probes.append(("fm-derivative", np.abs(np.diff(f))))
    a2 = np.abs(x).astype(np.float64) ** 2
    if a2.size > 512:
        probes.append(("squared-envelope", a2))

    best = None
    for name, probe in probes:
        got = _spectral_lines(probe, fs, f_lo, f_hi)
        if got is None:
            continue
        fr, S, noise = got
        fund = _fundamental(fr, S, noise)
        if fund is None:
            continue
        rate, snr = fund
        if best is None or snr > best["line_snr"]:
            best = {"symbol_rate": rate, "line_snr": snr, "method": name}
    return best

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.0))
demo = [("GFSK 250 kbps", gen_gfsk(2e6, 250e3, 60e3, 800), 2e6, 250e3, 246e3),
        ("QPSK RRC 5 Msym", gen_linear(20e6, 5e6, 2000), 20e6, 5e6, 5.3e6),
        ("PPM/OOK 1 Mchip", gen_ppm_ook(8e6, 1e6, 300), 8e6, 1e6, 5.1e6)]
for a, (title, x, fs, true_rs, bwh) in zip(ax, demo):
    x = awgn(x, 25)
    est = estimate_symbol_rate(x, fs, bw_hint=bwh)
    f = inst_freq(x, fs)
    probe = np.abs(np.diff(np.sign(f - np.median(f)))) if f.size > 512 else np.abs(x) ** 2
    got = _spectral_lines(probe, fs, bwh / 60, bwh * 1.6)
    if got:
        fr, S, nz = got
        a.semilogy(fr / 1e3, S / nz, lw=0.5)
        a.axvline(true_rs / 1e3, color="tab:red", ls="--", lw=1, label="true $R_s$")
        for m in (2, 3):
            a.axvline(true_rs * m / 1e3, color="tab:orange", ls=":", lw=0.8)
    a.set_title(f"{title}\nestimated {est['symbol_rate']/1e3:.0f} kHz "
                f"(true {true_rs/1e3:.0f}), SNR {est['line_snr']:.0f}x\n{est['method']}",
                fontsize=8)
    a.set_xlabel("cycle frequency (kHz)"); a.legend(fontsize=7)
ax[0].set_ylabel("line strength / median")
plt.tight_layout(); plt.show()

## 13. Counting FSK tones with an eye diagram

Given the symbol rate we can count modulation levels. The obvious approach —
histogram the smoothed discriminator — fails for narrowband 4-FSK, and §9
explains why: the smoothing needed to beat the noise is comparable to a symbol
period, so the window **straddles transitions** and averages the inner levels
back into each other.

Measured on DMR 4-FSK at 25 dB, sweeping the blanket window:

| window $W$ | peaks found | levels |
|---|---|---|
| 1 | 1 | noise-dominated blob |
| 3 | 3 | marginal |
| 5 | 2 | inner levels merged |
| 7, 10, 15 | 1 | fully merged |

There is no good choice. Maximising the peak count instead just rewards noise —
that variant reported 3, 4, and 5 tones on 2-FSK and QPSK signals.

### Sample the decision points instead

With $R_s$ known, $\mathrm{sps} = f_s/R_s$, and we sample the discriminator once
per symbol at phase $p$:

$$v_p[m] = \hat f\bigl[\,\mathrm{round}(p + m \cdot \mathrm{sps})\,\bigr],
\qquad p^\star = \arg\max_p \operatorname{Var}\bigl(v_p\bigr)$$

Maximum variance identifies the **open eye**: at the symbol centre the levels
are maximally separated, while at a transition they collapse toward the mean.
This is the classic timing-recovery criterion, used here for a different
purpose. The histogram of $v_{p^\star}$ separates all $M$ levels cleanly, and on
the DMR case it recovers all four tones where every blanket window failed.

### Validating a peak set

Peaks alone are not enough — a QPSK discriminator throws up accidental bumps
that formed a "uniform triplet" and were classified as 4-FSK. M-ary FSK tones
are *uniformly spaced* with *comparable occupancy*, so we require spacing
CV < 0.30 and a min/max peak-height ratio > 0.35, and additionally require a
near-constant envelope (CV < 0.25) before any FSK verdict.

In [ ]:
def _tone_peaks(v: np.ndarray, nbins: Optional[int] = None):
    """Histogram a set of frequency samples and return (peaks, prominences)."""
    v = v - np.median(v)
    if v.size < 40:
        return None
    if nbins is None:
        nbins = int(np.clip(int(np.sqrt(v.size) * 2), 33, 129))
    span = 2.2 * float(np.percentile(np.abs(v), 97)) or 1.0
    h, e = np.histogram(v, bins=np.linspace(-span, span, nbins))
    h = _smooth(h.astype(float), 3)
    if h.max() <= 0:
        return None
    h = h / h.max()
    c = 0.5 * (e[:-1] + e[1:])
    pk, props = signal.find_peaks(h, prominence=0.20, distance=2)
    if pk.size == 0:
        return None
    o = np.argsort(c[pk])
    return c[pk][o], h[pk][o]


def _validate_tones(tones: np.ndarray, heights: np.ndarray) -> bool:
    """M-FSK tones are uniformly spaced and of comparable occupancy."""
    if tones.size < 2:
        return False
    if heights.min() / heights.max() < 0.35:
        return False
    if tones.size >= 3:
        d = np.diff(tones)
        if np.std(d) / (np.mean(d) + 1e-30) > 0.30:
            return False
    return True


def fsk_tones(x: np.ndarray, fs: float, bw_hint: Optional[float] = None,
              symbol_rate: Optional[float] = None):
    """Count instantaneous-frequency modes -> 2-FSK / 4-FSK / M-FSK.

    Blanket-averaging the discriminator straddles symbol transitions and merges
    the inner levels of a 4-FSK signal back together. With a symbol rate in
    hand we can instead sample at the eye's decision points: the phase that
    maximises the sampled variance is the open eye, and the histogram of those
    samples separates all M levels cleanly. The blanket histogram is retained
    as a fallback for when the symbol rate is unknown or unreliable.

    A peak set is only accepted as M-FSK if it is uniformly spaced with
    comparable level occupancy — that is what separates real M-FSK from the
    accidental bumps a linear modulation's discriminator produces.
    """
    f_raw = inst_freq(x, fs)
    if f_raw.size < 256:
        return None
    span_hint = bw_hint if bw_hint else 2.0 * np.percentile(np.abs(f_raw), 98)

    cands = []   # (n_tones, tones, heights, method)

    if symbol_rate and symbol_rate > 0:
        sps = fs / symbol_rate
        if 2.0 <= sps <= 4096.0:
            # Sweep the pre-smoothing length rather than deriving one value. A
            # single derived window is fragile: on narrowband 4-FSK, w=round(sps/4)
            # resolved only 2 of the 4 levels while w=3 resolved all four, and
            # which one wins shifts with the noise realisation. Widening the set
            # is safe because selection below only accepts peak sets that are
            # uniformly spaced with comparable occupancy — a bad window yields an
            # invalid set and is discarded rather than competing.
            cand_w = {1, 2, 3, int(round(sps / 6)), int(round(sps / 4)),
                      int(round(sps / 3)), int(round(sps / 2))}
            for w in sorted(x for x in cand_w if 1 <= x <= min(16, max(1, sps - 1))):
                f = _smooth(f_raw, w)
                best = None
                for ph in range(max(1, int(round(sps)))):
                    idx = np.round(np.arange(ph, f.size - 1, sps)).astype(int)
                    idx = idx[idx < f.size]
                    if idx.size < 40:
                        continue
                    v = f[idx]
                    if best is None or v.var() > best[0]:
                        best = (v.var(), v)
                if best is None:
                    continue
                got = _tone_peaks(best[1])
                if got:
                    cands.append((got[0].size, got[0], got[1], f"eye(w={w})"))

    # blanket histogram fallback
    w = int(np.clip(round(0.5 * (fs / symbol_rate if symbol_rate else fs / max(span_hint, 1.0))), 1, 64))
    got = _tone_peaks(_smooth(f_raw, w), nbins=129)
    blanket = (got[0].size, got[0], got[1], f"blanket(w={w})") if got else None
    if blanket:
        cands.append(blanket)

    valid = [c for c in cands if c[0] in (2, 4, 8) and _validate_tones(c[1], c[2])]

    def _rank(c):
        """Prefer more levels, but only when those levels are well occupied.

        Sweeping windows means a noisy one can occasionally fake a uniform
        4-tone set on a genuine 2-FSK signal, and 'prefer larger M' alone would
        promote it. Requiring a min/max occupancy above 0.5 before a higher M
        outranks a clean 2-tone answer costs nothing on real M-ary FSK, whose
        levels are equiprobable by construction.
        """
        n, tones, heights, _ = c
        occ = float(heights.min() / heights.max())
        return (n if (n == 2 or occ > 0.5) else 0, occ)

    pick = max(valid, key=_rank) if valid else (blanket or (cands[0] if cands else None))
    if pick is None:
        return {"n_tones": 0, "tone_freqs": [], "deviation": float(np.std(f_raw)),
                "uniform": False, "method": "none"}

    n, tones, heights, method = pick
    # Re-derive validity rather than testing `pick in valid`: these candidates
    # are tuples containing ndarrays, so `in` falls back to element-wise `==`
    # and raises "truth value of an array is ambiguous" as soon as the list holds
    # more than one entry. It only ever appeared to work because `valid` was
    # usually a single item.
    pick_valid = bool(n in (2, 4, 8) and _validate_tones(tones, heights))
    d = np.diff(tones)
    dev = float(tones[1] - tones[0]) if n == 2 else (float(np.median(d)) if d.size else 0.0)
    return {"n_tones": int(n), "tone_freqs": tones.tolist(), "deviation": dev,
            "uniform": pick_valid, "method": method}


def estimate_pulse_rate(x: np.ndarray, fs: float):
    """Chip/pulse rate of an on-off keyed signal from envelope run lengths.

    Far more reliable than a spectral line for short pulsed frames: a 112-bit
    ADS-B burst is only ~1000 samples, which is too few bins for the
    cyclostationary method to beat its own noise.
    """
    a = np.abs(x).astype(np.float64)
    thr = 0.5 * float(np.percentile(a, 95))
    if thr <= 0:
        return None
    b = a > thr
    if b.all() or not b.any():
        return None
    idx = np.flatnonzero(np.diff(b.astype(np.int8)) != 0)
    if idx.size < 6:
        return None
    runs = np.diff(idx).astype(float)
    r = float(np.percentile(runs, 10))
    if r < 2.0:
        return None
    return {"symbol_rate": fs / r, "line_snr": 8.0, "method": "envelope-run-length"}

In [ ]:
fs, Rs, dev = 48e3, 4800.0, 1944.0
x = awgn(gen_gfsk(fs, Rs, dev, 900, bt=0.6, m=4), 25)
f = inst_freq(x, fs)
sps = fs / Rs

fig, ax = plt.subplots(1, 3, figsize=(13, 3.1))

for W in (1, 3, 5, 9, 15):
    fh = _smooth(f, W); fh = fh - np.median(fh)
    h, e = np.histogram(fh, bins=np.linspace(-6100, 6100, 129))
    ax[0].plot(0.5 * (e[:-1] + e[1:]), _smooth(h.astype(float), 5) / h.max(),
               lw=0.8, label=f"W={W}")
ax[0].set_title("blanket smoothing: no window works", fontsize=8)
ax[0].set_xlabel("Hz"); ax[0].legend(fontsize=7)

best = None
for ph in range(int(round(sps))):
    idx = np.round(np.arange(ph, f.size - 1, sps)).astype(int)
    idx = idx[idx < f.size]
    v = f[idx]
    if best is None or v.var() > best[0]:
        best = (v.var(), ph, v)
_, ph, v = best
tp = _tone_peaks(v)
ax[1].hist(v - np.median(v), bins=64, color="tab:blue", alpha=0.8)
for tone in tp[0]:
    ax[1].axvline(tone, color="tab:red", ls="--", lw=1)
for lvl in (-dev, -dev / 3, dev / 3, dev):
    ax[1].axvline(lvl, color="tab:green", ls=":", lw=1)
ax[1].set_title(f"eye sampling at phase {ph}: {tp[0].size} tones\n"
                "red=found, green=true levels", fontsize=8)
ax[1].set_xlabel("Hz")

varz = []
for p2 in range(int(round(sps))):
    idx = np.round(np.arange(p2, f.size - 1, sps)).astype(int)
    varz.append(f[idx[idx < f.size]].var())
ax[2].stem(range(len(varz)), varz)
ax[2].axvline(ph, color="tab:red", ls="--", lw=1, label=f"chosen p*={ph}")
ax[2].set_xlabel("sampling phase"); ax[2].set_title("variance identifies the open eye",
                                                    fontsize=8)
ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

r = fsk_tones(x, fs, bw_hint=6100, symbol_rate=Rs)
print(f"fsk_tones -> {r['n_tones']} tones via {r['method']}, "
      f"spacing {r['deviation']:.0f} Hz (true {2*dev/3:.0f} Hz), uniform={r['uniform']}")

# Part III — Classification

## 14. Assembling the feature vector

`extract_features` runs the whole measurement chain on one burst. Two ordering
decisions in it are load-bearing:

1. **Amplitude statistics are taken before filtering.** CV and PAPR describe the
   signal as received; measuring them after the anti-alias filter would report
   the filter's envelope, not the modulation's.
2. **Symbol rate is estimated before tone counting**, because §13's eye sampling
   needs $\mathrm{sps}$. This dependency is why the pipeline is not a simple
   left-to-right sequence of independent features.

In [ ]:
@dataclass
class Features:
    t0: float
    duration: float
    snr_db: float
    bw99: float
    bw_rms: float
    cfo: float
    envelope_cv: float
    papr_db: float
    flatness: float
    edge_db: float
    modclass: str = "unknown"
    symbol_rate: Optional[float] = None
    symbol_rate_snr: Optional[float] = None
    n_tones: Optional[int] = None
    fsk_deviation: Optional[float] = None
    tones_uniform: Optional[bool] = None
    decimation: int = 1
    ofdm_fft: Optional[int] = None
    ofdm_cp: Optional[int] = None
    ofdm_score: Optional[float] = None
    subcarrier_spacing: Optional[float] = None
    chirp_rate: Optional[float] = None
    chirp_score: Optional[float] = None
    css_bw: Optional[float] = None
    css_sf: Optional[int] = None
    notes: list[str] = field(default_factory=list)


def extract_features(x: np.ndarray, fs: float, burst: Burst, noise_db: float) -> Features:
    seg = x[burst.i0 : burst.i1]
    if seg.size < 64:
        raise ValueError("burst too short for analysis")

    f, pxx = welch_psd(seg, fs)
    bw99, center = occupied_bandwidth(f, pxx, 0.99)
    bwr = rms_bandwidth(f, pxx, bw=bw99, center=center)

    # De-rotate to the measured centre, then channel-filter and decimate to
    # ~4x the occupied bandwidth. Discarding out-of-band noise is the single
    # cheapest SNR win available (often ~9 dB on a narrowband burst in a wide
    # capture) and it makes sample-count-based smoothing windows meaningful.
    n = np.arange(seg.size)
    seg_c = seg * np.exp(-2j * np.pi * center * n / fs)
    fs_a = fs
    # Target ~6x the occupied bandwidth. Going tighter than that starves the
    # discriminator of samples per symbol, which costs more than the filtering
    # gains: at 4 samples/symbol a GFSK eye stops resolving reliably.
    decim = int(max(1, np.floor(fs / (6.0 * max(bw99, 1.0)))))
    if decim >= 3 and seg_c.size // decim >= 512:
        taps = min(129, (seg_c.size // 4) | 1)
        h = signal.firwin(taps, 1.0 / decim, window=("kaiser", 6.0))
        seg_c = signal.lfilter(h, 1.0, seg_c)[::decim].astype(np.complex64)
        fs_a = fs / decim
        ft_notes_decim = f"channel-filtered and decimated {decim}x -> {fs_a/1e6:.4g} Msps"
    else:
        decim, ft_notes_decim = 1, None

    a = np.abs(seg).astype(np.float64)
    cv = float(np.std(a) / (np.mean(a) + 1e-30))
    papr = 10.0 * math.log10((np.max(a) ** 2) / (np.mean(a**2) + 1e-30) + 1e-30)
    sig_db = 10.0 * math.log10(np.mean(a**2) + 1e-30)

    ft = Features(
        t0=burst.t0,
        duration=burst.duration,
        snr_db=sig_db - noise_db,
        bw99=bw99,
        bw_rms=bwr,
        cfo=center,
        envelope_cv=cv,
        papr_db=papr,
        flatness=spectral_flatness(pxx, f, bw99, center),
        edge_db=edge_sharpness(pxx, f, bw99, center),
    )

    # --- OFDM ---
    min_lag = max(8, int(fs_a / max(bw99, 1.0) * 4))
    ac = cp_autocorrelation(seg_c, min_lag=min_lag, max_lag=4096)
    if ac:
        ft.ofdm_score = ac["score"]
        if ac["score"] > 10.0 and ac["value"] > 0.03 and ac["lag"] >= 16:
            ft.ofdm_fft = ac["lag"]
            ft.ofdm_cp = estimate_cp_length(seg_c, ac["lag"])
            ft.subcarrier_spacing = fs_a / ac["lag"]

    # --- chirp / CSS ---
    ch = detect_chirp(seg_c, fs_a, bw_hint=bw99)
    if ch:
        ft.chirp_rate = ch["chirp_rate"]
        ft.chirp_score = ch["score"]
        ft.css_bw = ch["bw"]
        ft.css_sf = ch["sf"]

    # --- symbol / chip rate (needed before tone counting) ---
    sr = estimate_symbol_rate(seg_c, fs_a, bw_hint=bw99)
    if cv > 0.60:                                   # pulsed: trust run lengths
        pr = estimate_pulse_rate(seg_c, fs_a)
        if pr and (sr is None or pr["line_snr"] >= (sr["line_snr"] or 0) * 0.5):
            sr = pr
    if sr:
        ft.symbol_rate = sr["symbol_rate"]
        ft.symbol_rate_snr = sr["line_snr"]
        ft.notes.append(f"symbol rate from {sr['method']}")

    # --- FSK tones ---
    tn = fsk_tones(seg_c, fs_a, bw_hint=bw99, symbol_rate=ft.symbol_rate)
    if tn:
        ft.n_tones = tn["n_tones"]
        ft.fsk_deviation = tn["deviation"]
        ft.tones_uniform = tn.get("uniform")
        ft.notes.append(f"tone count via {tn.get('method')}")

    if ft_notes_decim:
        ft.notes.append(ft_notes_decim)
    ft.decimation = decim
    ft.modclass = _decide_modclass(ft)
    return ft

## 15. Deciding the modulation family

An ordered decision list rather than a trained classifier — each test is
something we derived, so a wrong answer is traceable to a specific measurement.
Order matters: CSS is tested first because a matched dechirp is nearly
unambiguous, and OFDM before OOK because a large envelope swing alone does not
distinguish them.

In [ ]:
def _decide_modclass(ft: Features) -> str:
    """Coarse modulation family from the measured features."""
    # CSS first: a matched dechirp is close to unambiguous
    if ft.chirp_score is not None and ft.chirp_score > 0.15 and ft.envelope_cv < 0.45:
        return "css"

    # OFDM: cyclic-prefix autocorrelation + varying envelope + flat spectrum
    if ft.ofdm_fft and ft.envelope_cv > 0.30 and ft.flatness > 0.25:
        return "ofdm"

    # Pulsed / OOK: large envelope swing with no CP structure
    if ft.envelope_cv > 0.70 and ft.papr_db > 2.0 and not ft.ofdm_fft:
        return "ook-pulsed"

    const_env = ft.envelope_cv < 0.22
    tones_ok = bool(ft.tones_uniform) and ft.n_tones is not None

    if const_env or (ft.envelope_cv < 0.25 and tones_ok):
        if tones_ok and ft.n_tones == 2:
            return "fsk2"
        if tones_ok and ft.n_tones in (3, 4):
            return "fsk4"
        if tones_ok and ft.n_tones and ft.n_tones > 4:
            return "fsk-multi"
        if const_env:
            return "const-env-phase"        # PSK / MSK / GMSK / OQPSK / analog FM

    return "linear-shaped"                  # PSK / QAM with pulse shaping

## 16. The protocol database

Roughly 35 entries, each with the bandwidth, symbol rate, burst duration,
subcarrier spacing, and deviation ranges its standard permits — plus the
specification reference, so any classification can be checked against the
document.

`band` is the most powerful field. Given a tuner centre frequency it acts as a
hard gate, and without it several protocols are simply not separable: DMR and
P25 Phase 1 share bandwidth *and* symbol rate *and* modulation, differing only
in allocation and vocoder.

In [ ]:
MHz = 1e6


kHz = 1e3


PROTOCOLS: list[dict] = [
    # ---- 2.4 GHz ISM ----
    dict(name="Wi-Fi 802.11b/g (DSSS/CCK)", mod=["dsss-psk", "linear-shaped", "const-env-phase"],
         bw=(14 * MHz, 24 * MHz), rate=(0.9e6, 1.5e6), dur=(20e-6, 5e-3),
         band=[(2.40e9, 2.50e9)], spec="IEEE 802.11-2020 cl.16/17"),
    dict(name="Wi-Fi 802.11g/n/ax 20 MHz (OFDM)", mod=["ofdm"],
         bw=(15 * MHz, 20.5 * MHz), rate=None, dur=(20e-6, 6e-3),
         band=[(2.40e9, 2.50e9), (5.15e9, 5.90e9), (5.925e9, 7.125e9)],
         scs=(300 * kHz, 330 * kHz), spec="IEEE 802.11-2020 cl.17/19/27"),
    dict(name="Wi-Fi 802.11n/ac/ax 40 MHz", mod=["ofdm"],
         bw=(33 * MHz, 41 * MHz), rate=None, dur=(20e-6, 6e-3),
         band=[(2.40e9, 2.50e9), (5.15e9, 7.125e9)],
         scs=(300 * kHz, 330 * kHz), spec="IEEE 802.11-2020"),
    dict(name="Wi-Fi 802.11ac/ax 80 MHz", mod=["ofdm"],
         bw=(70 * MHz, 82 * MHz), rate=None, dur=(20e-6, 6e-3),
         band=[(5.15e9, 7.125e9)], scs=(300 * kHz, 330 * kHz), spec="IEEE 802.11-2020"),
    dict(name="Bluetooth Classic (GFSK, BR)", mod=["fsk2", "const-env-phase"],
         bw=(0.7 * MHz, 1.3 * MHz), rate=(0.85e6, 1.15e6), dur=(100e-6, 3.0e-3),
         band=[(2.402e9, 2.480e9)], dev=(140 * kHz, 175 * kHz),
         spec="Bluetooth Core 5.4, Vol 6"),
    dict(name="Bluetooth LE 1M (GFSK)", mod=["fsk2", "const-env-phase"],
         bw=(0.8 * MHz, 1.4 * MHz), rate=(0.9e6, 1.1e6), dur=(40e-6, 400e-6),
         band=[(2.402e9, 2.480e9)], dev=(225 * kHz, 275 * kHz),
         spec="Bluetooth Core 5.4, Vol 6 Part B"),
    dict(name="Bluetooth LE 2M (GFSK)", mod=["fsk2", "const-env-phase"],
         bw=(1.6 * MHz, 2.6 * MHz), rate=(1.8e6, 2.2e6), dur=(25e-6, 250e-6),
         band=[(2.402e9, 2.480e9)], dev=(450 * kHz, 550 * kHz),
         spec="Bluetooth Core 5.4"),
    dict(name="Zigbee / 802.15.4 O-QPSK DSSS", mod=["const-env-phase", "linear-shaped"],
         bw=(1.8 * MHz, 3.0 * MHz), rate=(1.8e6, 2.2e6), dur=(200e-6, 4.5e-3),
         band=[(2.405e9, 2.485e9)], spec="IEEE 802.15.4-2020 cl.12"),
    dict(name="Nordic/ANT-class 1 Mbps GFSK", mod=["fsk2"],
         bw=(0.8 * MHz, 1.4 * MHz), rate=(0.9e6, 1.1e6), dur=(30e-6, 500e-6),
         band=[(2.400e9, 2.485e9)], dev=(140 * kHz, 200 * kHz),
         spec="ANT Message Protocol / nRF24 datasheet"),

    # ---- sub-GHz ISM ----
    dict(name="LoRa / CSS (LoRaWAN, Meshtastic, ELRS)", mod=["css"],
         bw=(100 * kHz, 550 * kHz), rate=None, dur=(1e-3, 3.0),
         band=[(150e6, 960e6), (2.400e9, 2.485e9)],
         spec="LoRa Alliance regional params; PHY is Semtech proprietary"),
    dict(name="SiK / MAVLink telemetry (2-FSK/GFSK)", mod=["fsk2"],
         bw=(60 * kHz, 400 * kHz), rate=(30e3, 260e3), dur=(1e-3, 60e-3),
         band=[(410e6, 480e6), (863e6, 928e6), (2.400e9, 2.485e9)],
         dev=(20 * kHz, 130 * kHz), spec="SiK firmware (Si1000/Si4432), MAVLink v2"),
    dict(name="Z-Wave (G.9959 GFSK)", mod=["fsk2"],
         bw=(30 * kHz, 400 * kHz), rate=(9e3, 110e3), dur=(1e-3, 40e-3),
         band=[(865e6, 927e6)], spec="ITU-T G.9959"),
    dict(name="Wi-SUN / 802.15.4g 2-FSK", mod=["fsk2"],
         bw=(100 * kHz, 800 * kHz), rate=(45e3, 310e3), dur=(1e-3, 50e-3),
         band=[(470e6, 960e6)], spec="IEEE 802.15.4g / Wi-SUN FAN 1.1"),
    dict(name="Generic ISM OOK remote (rtl_433 class)", mod=["ook-pulsed"],
         bw=(10 * kHz, 400 * kHz), rate=(200.0, 20e3), dur=(0.5e-3, 200e-3),
         band=[(300e6, 450e6), (860e6, 930e6)], spec="device-specific; see rtl_433"),
    dict(name="Sigfox uplink (DBPSK, UNB)", mod=["linear-shaped", "const-env-phase"],
         bw=(50.0, 400.0), rate=(90.0, 650.0), dur=(1.0, 3.0),
         band=[(862e6, 928e6)], spec="Sigfox radio specification"),
    dict(name="mioty (TS-UNB telegram splitting)", mod=["fsk2", "const-env-phase"],
         bw=(1 * kHz, 30 * kHz), rate=(2e3, 6e3), dur=(0.3, 2.0),
         band=[(862e6, 928e6)], spec="ETSI TS 103 357"),

    # ---- aviation / maritime ----
    # PPM encodes each 1 us chip as a 0.5 us pulse, so a run-length estimator
    # legitimately reports ~2 MHz rather than the 1 Mchip/s frame rate. The window
    # spans both readings; burst duration is the feature that actually separates a
    # 120 us extended squitter from a 20 us Mode A/C/S reply.
    dict(name="ADS-B 1090ES (PPM/OOK)", mod=["ook-pulsed"],
         bw=(1.0 * MHz, 8.0 * MHz), rate=(0.9e6, 2.6e6), dur=(50e-6, 130e-6),
         band=[(1089e6, 1091e6)], spec="RTCA DO-260B / ICAO Annex 10"),
    dict(name="Mode A/C/S interrogation & reply", mod=["ook-pulsed"],
         bw=(1.0 * MHz, 8.0 * MHz), rate=(0.5e6, 2.2e6), dur=(15e-6, 50e-6),
         band=[(1029e6, 1091e6)], spec="ICAO Annex 10 Vol IV"),
    dict(name="AIS (GMSK 9.6 kbps)", mod=["const-env-phase", "fsk2"],
         bw=(9 * kHz, 25 * kHz), rate=(9e3, 10.2e3), dur=(20e-3, 30e-3),
         band=[(161.9e6, 162.1e6), (156e6, 163e6)], spec="ITU-R M.1371-5"),
    dict(name="FLARM / OGN (GFSK 100 kbps)", mod=["fsk2"],
         bw=(150 * kHz, 400 * kHz), rate=(90e3, 110e3), dur=(0.3e-3, 2e-3),
         band=[(868e6, 869e6)], spec="FLARM DDL (partly reverse-engineered)"),

    # ---- LMR / paging ----
    dict(name="DMR (4-FSK, 12.5 kHz)", mod=["fsk4"],
         bw=(8 * kHz, 14 * kHz), rate=(4.5e3, 5.1e3), dur=(20e-3, 0.5),
         band=[(130e6, 950e6)], spec="ETSI TS 102 361"),
    dict(name="P25 Phase 1 (C4FM)", mod=["fsk4"],
         bw=(8 * kHz, 14 * kHz), rate=(4.5e3, 5.1e3), dur=(20e-3, 5.0),
         band=[(130e6, 900e6)], spec="TIA-102.BAAA"),
    dict(name="TETRA (pi/4-DQPSK)", mod=["linear-shaped"],
         bw=(20 * kHz, 30 * kHz), rate=(17e3, 19e3), dur=(10e-3, 5.0),
         band=[(380e6, 470e6)], spec="ETSI EN 300 392-2"),
    dict(name="POCSAG paging (2-FSK)", mod=["fsk2"],
         bw=(6 * kHz, 25 * kHz), rate=(480.0, 2500.0), dur=(50e-3, 5.0),
         band=[(130e6, 960e6)], spec="ITU-R M.584"),
    dict(name="Analog FM voice / NBFM", mod=["fsk-multi", "const-env-phase"],
         bw=(8 * kHz, 20 * kHz), rate=None, dur=(50e-3, 60.0),
         band=[(25e6, 1300e6)], spec="n/a (analog)"),

    # ---- cellular / cordless ----
    dict(name="LTE / 5G NR downlink (OFDM, 15 kHz SCS)", mod=["ofdm"],
         bw=(1.2 * MHz, 100 * MHz), rate=None, dur=(0.5e-3, 60.0),
         band=[(400e6, 6.0e9)], scs=(13 * kHz, 17 * kHz),
         spec="3GPP TS 36.211 / 38.211"),
    dict(name="5G NR (OFDM, 30 kHz SCS)", mod=["ofdm"],
         bw=(5 * MHz, 100 * MHz), rate=None, dur=(0.25e-3, 60.0),
         band=[(400e6, 7.2e9)], scs=(27 * kHz, 33 * kHz), spec="3GPP TS 38.211"),
    dict(name="GSM / GMSK burst", mod=["const-env-phase", "fsk2"],
         bw=(180 * kHz, 300 * kHz), rate=(265e3, 275e3), dur=(400e-6, 700e-6),
         band=[(380e6, 1.99e9)], spec="3GPP TS 45.004"),
    dict(name="DECT (GFSK 1.152 Msym)", mod=["fsk2", "const-env-phase"],
         bw=(1.0 * MHz, 1.9 * MHz), rate=(1.05e6, 1.25e6), dur=(300e-6, 500e-6),
         band=[(1.88e9, 1.93e9)], spec="ETSI EN 300 175-2"),

    # ---- broadcast ----
    dict(name="DVB-T/T2 (OFDM)", mod=["ofdm"],
         bw=(6.5 * MHz, 8.2 * MHz), rate=None, dur=(1e-3, 60.0),
         band=[(470e6, 862e6)], scs=(0.9e3, 4.5e3), spec="ETSI EN 300 744 / 302 755"),
    dict(name="DAB+ (OFDM, 1 kHz SCS)", mod=["ofdm"],
         bw=(1.4 * MHz, 1.6 * MHz), rate=None, dur=(1e-3, 60.0),
         band=[(174e6, 240e6)], scs=(0.8e3, 1.2e3), spec="ETSI EN 300 401"),
    dict(name="FM broadcast (WBFM)", mod=["fsk-multi", "const-env-phase"],
         bw=(150 * kHz, 260 * kHz), rate=None, dur=(50e-3, 60.0),
         band=[(87e6, 109e6)], spec="ITU-R BS.450"),
]

## 17. Scoring

### Per-feature range score

Measurements decay smoothly outside the permitted range, in octaves:

$$s_i = \max\!\left(0,\; 1 - \frac{\bigl|\log_2(v/v_{\text{ref}})\bigr|}{\text{tol}}\right)$$

Log-domain because RF quantities span decades: being 50 kHz off means something
entirely different at 100 kHz than at 20 MHz.

### Combination: gate times weighted geometric mean

$$\text{score} \;=\; g_{\text{mod}} \cdot \exp\!\left(\frac{\sum_i w_i \ln s_i}{\sum_i w_i}\right)$$

The **geometric** mean is the important choice. An arithmetic mean lets a
candidate survive one decisive contradiction by scoring well elsewhere; the
geometric mean sends the product toward zero if any single $s_i$ does, which
matches how the reasoning should work — a signal whose bandwidth is wrong by a
factor of eight is not that protocol, however well everything else fits. It is
also equivalent to a log-likelihood sum under a naive-Bayes-style independence
assumption.

Modulation family enters as a **multiplicative gate**, not a term in the mean.
As one term among several, a mismatched family was survivable: an OFDM-only
entry (5G NR) scored 50% on a QPSK signal. As a gate the same case scores 20%,
below the confidence floor.

### Confidence floor

A leader at 20% means *nothing in the database fits*, not *probably this one*.
Below `CONFIDENCE_FLOOR = 0.35` the report says so explicitly rather than naming
a protocol.

In [ ]:
MOD_COMPAT = {
    # detected -> {db family: score}
    "ofdm": {"ofdm": 1.0, "linear-shaped": 0.15},
    "css": {"css": 1.0},
    "fsk2": {"fsk2": 1.0, "const-env-phase": 0.55, "fsk4": 0.15, "fsk-multi": 0.3},
    "fsk4": {"fsk4": 1.0, "fsk-multi": 0.6, "fsk2": 0.25, "const-env-phase": 0.4},
    "fsk-multi": {"fsk-multi": 1.0, "fsk4": 0.5, "fsk2": 0.3, "const-env-phase": 0.5},
    "const-env-phase": {"const-env-phase": 1.0, "fsk2": 0.55, "fsk4": 0.4,
                        "linear-shaped": 0.5, "dsss-psk": 0.6, "fsk-multi": 0.4},
    "linear-shaped": {"linear-shaped": 1.0, "dsss-psk": 0.8,
                      "const-env-phase": 0.5, "ofdm": 0.2},
    "ook-pulsed": {"ook-pulsed": 1.0, "linear-shaped": 0.15},
    "unknown": {},
}


def _range_score(value: Optional[float], rng, tol_octaves: float = 1.0) -> Optional[float]:
    """1.0 inside the range, decaying log-linearly outside it."""
    if value is None or rng is None:
        return None
    lo, hi = rng
    if lo <= value <= hi:
        return 1.0
    ref = lo if value < lo else hi
    if value <= 0 or ref <= 0:
        return 0.0
    octaves = abs(math.log2(value / ref))
    return float(max(0.0, 1.0 - octaves / tol_octaves))


def score_protocol(ft: Features, proto: dict, rf_center_hz: Optional[float]) -> dict:
    reasons: list[str] = []
    terms: list[tuple[float, float]] = []   # (score, weight)

    # band gate — hard filter when the tuner frequency is known
    if rf_center_hz is not None:
        ok = any(lo <= rf_center_hz <= hi for lo, hi in proto["band"])
        if not ok:
            return dict(name=proto["name"], score=0.0,
                        reasons=["RF centre outside allocated band"], spec=proto["spec"])

    m = MOD_COMPAT.get(ft.modclass, {})
    mod_gate = max((m.get(k, 0.0) for k in proto["mod"]), default=0.0)
    reasons.append(f"modulation {ft.modclass} vs {'/'.join(proto['mod'])}: "
                   f"gate x{mod_gate:.2f}")
    if mod_gate <= 0.0:
        return dict(name=proto["name"], score=0.0, reasons=reasons, spec=proto["spec"])

    bw_s = _range_score(ft.bw99, proto["bw"], tol_octaves=1.0)
    if bw_s is not None:
        terms.append((bw_s, 3.0))
        reasons.append(f"BW99 {_hz(ft.bw99)} vs {_hz(proto['bw'][0])}–{_hz(proto['bw'][1])}: {bw_s:.2f}")

    if proto.get("rate") and ft.symbol_rate and (ft.symbol_rate_snr or 0) > 4.0:
        rs = _range_score(ft.symbol_rate, proto["rate"], tol_octaves=1.2)
        terms.append((rs, 2.0))
        reasons.append(f"symbol rate {_hz(ft.symbol_rate)} vs "
                       f"{_hz(proto['rate'][0])}–{_hz(proto['rate'][1])}: {rs:.2f}")

    if proto.get("dur"):
        # Weight 1.5, above its original 1.0. Frame durations are tightly
        # specified (an ADS-B extended squitter is exactly 112 bits = 120 us)
        # whereas a run-length chip-rate estimate on a pulsed frame is noisy, so
        # duration deserves more say than rate when the two disagree. This is
        # what separates ADS-B from a Mode A/C/S reply: same band, same
        # modulation, same bandwidth, different frame length.
        ds = _range_score(ft.duration, proto["dur"], tol_octaves=2.5)
        terms.append((ds, 1.5))
        reasons.append(f"burst {ft.duration*1e3:.3f} ms: {ds:.2f}")

    if proto.get("scs") and ft.subcarrier_spacing:
        ss = _range_score(ft.subcarrier_spacing, proto["scs"], tol_octaves=0.6)
        terms.append((ss, 2.5))
        reasons.append(f"subcarrier spacing {_hz(ft.subcarrier_spacing)}: {ss:.2f}")

    if proto.get("dev") and ft.fsk_deviation and ft.modclass.startswith("fsk"):
        vs = _range_score(ft.fsk_deviation, proto["dev"], tol_octaves=1.0)
        terms.append((vs, 1.5))
        reasons.append(f"FSK deviation {_hz(ft.fsk_deviation)}: {vs:.2f}")

    # weighted geometric mean -> one bad feature is punished hard
    num = sum(w * math.log(max(s, 1e-3)) for s, w in terms)
    den = sum(w for _, w in terms) or 1.0
    score = math.exp(num / den) * mod_gate
    return dict(name=proto["name"], score=float(score),
                reasons=reasons, spec=proto["spec"])

In [ ]:
CONFIDENCE_FLOOR = 0.35


@dataclass
class BurstResult:
    features: Features
    candidates: list[dict]
    preamble: Optional[object] = None      # a PreambleHit, if one matched

    @property
    def best(self) -> Optional[dict]:
        return self.candidates[0] if self.candidates else None

    @property
    def confident(self) -> bool:
        """True only when the leader clears the floor. A 20% leader means
        'nothing in the database fits', not 'probably this one'.

        A preamble correlation is confident by construction: the threshold was
        chosen for a specific false-alarm probability, so clearing it already
        *is* the confidence statement.
        """
        if self.preamble is not None:
            return True
        return bool(self.best and self.best["score"] >= CONFIDENCE_FLOOR)


def identify(
    x: np.ndarray,
    fs: float,
    rf_center_hz: Optional[float] = None,
    max_bursts: int = 12,
    top: int = 3,
    preamble_bank: Optional[object] = None,
    preamble_snr_tol_db: float = 12.0,
    **burst_kw,
) -> list[BurstResult]:
    """Segment ``x`` into bursts and rank protocol candidates for each.

    ``preamble_bank`` is any object exposing ``scan(x, fs, rf_center_hz)`` and
    returning hits with ``.name``, ``.t0``, ``.rho``, ``.threshold``,
    ``.n_template``, ``.spec`` and ``.est_snr_db`` — see ``iq_preamble``. It is
    duck-typed rather than imported so the blind classifier stays standalone.

    A preamble hit overrides the blind ranking rather than merging with it. The
    two are not comparable evidence: the blind score is a similarity heuristic,
    while a correlation above threshold is a statement about false-alarm
    probability. Where the correlator fires, its answer wins; the blind features
    are kept alongside as corroboration and as the source of the measurements
    (bandwidth, symbol rate, deviation) that the correlator does not produce.
    """
    x = np.asarray(x, dtype=np.complex64)
    bursts, noise_db = detect_bursts(x, fs, **burst_kw)

    # analyse the strongest / longest bursts first
    bursts = sorted(bursts, key=lambda b: -(b.i1 - b.i0))[:max_bursts]
    bursts = sorted(bursts, key=lambda b: b.i0)

    out = []
    for b in bursts:
        try:
            ft = extract_features(x, fs, b, noise_db)
        except ValueError:
            continue
        cands = [score_protocol(ft, p, rf_center_hz) for p in PROTOCOLS]
        cands = [c for c in cands if c["score"] > 0.02]
        cands.sort(key=lambda c: -c["score"])
        out.append(BurstResult(features=ft, candidates=cands[:top]))

    if preamble_bank is None:
        return out

    for hit in preamble_bank.scan(x, fs, rf_center_hz=rf_center_hz):
        target = None
        for r in out:
            t0, t1 = r.features.t0, r.features.t0 + r.features.duration
            if t0 - 200e-6 <= hit.t0 <= t1 + 200e-6:
                target = r
                break

        # Consistency gate. Equation (2) says a genuinely matched template must
        # return rho ~= sqrt(gamma/(1+gamma)), so its implied SNR should track
        # the SNR we measured on the burst. A template that correlates with a
        # frame's random payload rather than its preamble clears the noise-only
        # threshold but implies an SNR far below what the burst actually has —
        # which is how a BLE 2M template finds a 'preamble' inside a 30 dB BLE
        # 1M payload. One-sided on purpose: a hit is only ever rejected for
        # implying *less* signal than is measurably present.
        # Only applied to bursts whose SNR is high enough for the measurement
        # itself to mean anything. Near the detector's own floor the energy
        # detector reports whatever a noise fluctuation happened to do, and
        # letting that veto a correlation that cleared its threshold throws away
        # the entire point of the matched filter — it cost ~12 dB of POCSAG
        # sensitivity before this guard was added.
        if (target is not None and np.isfinite(target.features.snr_db)
                and target.features.snr_db > 10.0):
            if hit.est_snr_db < target.features.snr_db - preamble_snr_tol_db:
                target.features.notes.append(
                    f"rejected {hit.name} preamble hit: implied SNR "
                    f"{hit.est_snr_db:+.1f} dB inconsistent with measured "
                    f"{target.features.snr_db:+.1f} dB")
                continue
        if target is None:
            # Below roughly 6 dB the energy detector finds nothing, so a hit
            # with no matching burst is the normal low-SNR case, not an error.
            target = BurstResult(
                features=Features(
                    t0=hit.t0, duration=hit.template_duration,
                    snr_db=hit.est_snr_db, bw99=0.0, bw_rms=0.0, cfo=hit.cfo_hz,
                    envelope_cv=float("nan"), papr_db=float("nan"),
                    flatness=float("nan"), edge_db=float("nan"),
                    modclass="(preamble only — no burst detected)",
                    notes=["no energy-detected burst; identified by correlation alone"],
                ),
                candidates=[],
            )
            out.append(target)
        # keep the strongest hit per burst; hits arrive sorted by margin, so a
        # later one only wins if it is actually better
        if target.preamble is not None and \
                getattr(hit, "margin", 0) <= getattr(target.preamble, "margin", 0):
            continue
        target.preamble = hit
        target.candidates = [dict(
            name=hit.name, score=1.0, spec=hit.spec,
            reasons=[f"preamble correlation rho={hit.rho:.3f} vs threshold "
                     f"{hit.threshold:.3f} over N={hit.n_template} samples",
                     f"processing gain {hit.processing_gain_db:.1f} dB, "
                     f"CFO {hit.cfo_hz/1e3:+.1f} kHz",
                     f"implied per-sample SNR {hit.est_snr_db:+.1f} dB"],
        )] + [c for c in target.candidates if c["name"] != hit.name][: max(0, top - 1)]

    out.sort(key=lambda r: r.features.t0)
    return out

In [ ]:
def _hz(v: Optional[float]) -> str:
    if v is None:
        return "—"
    a = abs(v)
    for div, unit in ((1e9, "GHz"), (1e6, "MHz"), (1e3, "kHz")):
        if a >= div:
            return f"{v/div:.4g} {unit}"
    return f"{v:.4g} Hz"


def format_report(results: Sequence[BurstResult], verbose: bool = False) -> str:
    lines = []
    if not results:
        return "No bursts detected."
    for i, r in enumerate(results):
        ft = r.features
        lines.append(f"\n─── burst {i}  t={ft.t0*1e3:.3f} ms  "
                     f"len={ft.duration*1e3:.3f} ms  SNR≈{ft.snr_db:.1f} dB ───")
        lines.append(f"  class          {ft.modclass}")
        lines.append(f"  BW99 / RMS     {_hz(ft.bw99)} / {_hz(ft.bw_rms)}"
                     f"   offset {_hz(ft.cfo)}")
        lines.append(f"  env CV / PAPR  {ft.envelope_cv:.3f} / {ft.papr_db:.1f} dB"
                     f"   flatness {ft.flatness:.2f}  edge {ft.edge_db:.1f} dB")
        if ft.symbol_rate:
            lines.append(f"  symbol rate    {_hz(ft.symbol_rate)} "
                         f"(line SNR {ft.symbol_rate_snr:.1f}x)")
        if ft.n_tones:
            lines.append(f"  FSK tones      {ft.n_tones}, deviation {_hz(ft.fsk_deviation)}")
        if ft.ofdm_fft:
            lines.append(f"  OFDM           FFT≈{ft.ofdm_fft} samp, CP≈{ft.ofdm_cp}, "
                         f"SCS≈{_hz(ft.subcarrier_spacing)} (z={ft.ofdm_score:.1f})")
        if ft.chirp_score and ft.chirp_score > 0.10:
            lines.append(f"  CSS            BW≈{_hz(ft.css_bw)}, SF≈{ft.css_sf}, "
                         f"{ft.chirp_rate/1e6:.4g} MHz/s, "
                         f"dechirp energy {ft.chirp_score:.2f}")
        if not r.candidates:
            lines.append("  no candidate matched (unknown / not in database)")
            continue
        if r.preamble is not None:
            lines.append(f"  PREAMBLE MATCH  {r.preamble}")
        for note in ft.notes:
            lines.append(f"  · {note}")
        lines.append("  candidates:" if r.confident else
                     "  NO CONFIDENT MATCH — closest entries, all weak:")
        for c in r.candidates:
            lines.append(f"    {c['score']*100:5.1f}%  {c['name']}")
            lines.append(f"            spec: {c['spec']}")
            if verbose:
                for why in c["reasons"]:
                    lines.append(f"            · {why}")
    return "\n".join(lines)


def to_json(results: Sequence[BurstResult]) -> str:
    payload = []
    for r in results:
        d = asdict(r.features)
        d["candidates"] = r.candidates
        payload.append(d)
    return json.dumps(payload, indent=2, default=float)

# Part IV — Validating the blind path

## 18. Eight synthetic protocols, over many noise realisations

Ground truth for both the modulation family and the protocol name.

**A methodological point that matters more than the code.** This suite was
originally run at one fixed seed and reported 8/8. Re-running it across twelve
independent noise realisations showed the DMR 4-FSK case passing at *only that
seed* and failing at six others. A single-seed pass rate is not a measurement —
it is one sample of a random variable, and on a marginal case it is roughly a
coin flip.

Widening to a multi-seed sweep exposed four defects that the single-seed run had
concealed:

1. **Burst-detector time constants were absolute, not relative.** `win_s=2e-6`
   is 0.1 samples at 48 ksps, so the power envelope went unsmoothed and gap
   merging was disabled, fragmenting one burst into pieces. The largest fragment
   then had too few symbols for the eye histogram. Fixed with floors in samples
   (§5).
2. **A single derived smoothing window is fragile.** For narrowband 4-FSK,
   `round(sps/4)` resolved 2 of 4 levels while `w=3` resolved all four — and
   which one won shifted with the noise. Fixed by sweeping windows and letting
   the validation rules discard bad ones (§13).
3. **A latent `numpy` identity-vs-equality bug.** `pick in valid`, where the
   candidates are tuples containing arrays, falls back to element-wise `==` and
   raises *"truth value of an array is ambiguous"*. It had only ever appeared to
   work because `valid` usually held a single entry; widening the window sweep
   made multiple valid candidates common and it began throwing on six of eight
   cases. The exception was being swallowed by an `except ValueError` intended
   for short bursts — so the symptom was silently empty results, not a crash.
4. **The DMR generator was unrealistic.** C4FM is not Gaussian-filtered; BT=0.6
   over a 4-symbol span smeared the four levels well beyond what the standard
   implies. The test was measuring the generator, not the detector.

Current result: **94/96** across 12 seeds. The residual failures are the ADS-B
case being classified as a Mode A/C/S reply on 2 of 12 seeds — see §26.

In [ ]:
def cases():
    yield ("SiK/MAVLink 250 kbps GFSK, dev 60 kHz", 2e6, 915e6,
           pad(gen_gfsk(2e6, 250e3, 60e3, 700), 2e6, 1e-3, 1e-3),
           "fsk2", "SiK")

    yield ("BLE 1M GFSK, dev 250 kHz", 8e6, 2.44e9,
           pad(gen_gfsk(8e6, 1e6, 250e3, 300), 8e6),
           "fsk2", "Bluetooth")

    yield ("Wi-Fi 20 MHz OFDM (N=64, CP=16, 52 active)", 20e6, 2.437e9,
           pad(gen_ofdm(20e6, 64, 16, 52, 400), 20e6, 40e-6, 40e-6),
           "ofdm", "Wi-Fi")

    yield ("LTE-like OFDM, 15 kHz SCS, 10 MHz", 15.36e6, 1.85e9,
           pad(gen_ofdm(15.36e6, 1024, 72, 600, 60), 15.36e6, 200e-6, 200e-6),
           "ofdm", "LTE")

    yield ("LoRa BW=125 kHz SF7", 500e3, 868e6,
           pad(gen_lora(500e3, 125e3, 7, 40), 500e3, 4e-3, 4e-3),
           "css", "LoRa")

    yield ("ADS-B 1090ES PPM (112-bit frame)", 8e6, 1090e6,
           pad(gen_ppm_ook(8e6, 1e6, 120), 8e6, 60e-6, 60e-6),
           "ook-pulsed", "ADS-B")

    # C4FM is not Gaussian-filtered; it uses a shaping filter that keeps the eye
    # open at symbol centres. BT=0.6 over a 4-symbol Gaussian span smeared the
    # four levels well beyond anything the standard implies, making this case
    # pass or fail on the noise realisation rather than on detector quality.
    yield ("DMR 4-FSK 4.8 ksym, dev 1.944 kHz", 48e3, 446e6,
           pad(gen_gfsk(48e3, 4800, 1944, 900, bt=1.0, m=4), 48e3, 20e-3, 20e-3),
           "fsk4", "DMR")

    yield ("QPSK 5 Msym RRC a=0.35", 20e6, 2.4e9,
           pad(gen_linear(20e6, 5e6, 2000), 20e6, 40e-6, 40e-6),
           "linear-shaped", None)

In [ ]:
def run_blind_validation():
    passed = failed = 0
    for label, fs, fc, iq, want_mod, want_name in cases():
        res = identify(iq, fs, rf_center_hz=fc, top=3)
        print("=" * 78)
        print(f"{label}\n  fs={fs/1e6:g} MHz  fc={fc/1e6:g} MHz  "
              f"{len(iq)} samples  ->  {len(res)} burst(s)")
        if not res:
            print("  FAIL: no burst detected")
            failed += 1
            continue
        # take the longest burst as the verdict
        r = max(res, key=lambda q: q.features.duration)
        ft, best = r.features, r.best
        got_mod = ft.modclass
        got_name = best["name"] if best else "(no candidate)"
        ok_mod = got_mod == want_mod
        ok_name = want_name is None or (best and want_name.lower() in got_name.lower())
        print(f"  modclass  : {got_mod:16s} expected {want_mod:16s} {'OK' if ok_mod else 'MISMATCH'}")
        sc = best["score"] * 100 if best else 0.0
        print(f"  top match : {got_name}  ({sc:.1f}%)   {'OK' if ok_name else 'MISMATCH'}")
        print(f"  BW99={ft.bw99/1e3:.1f} kHz  Rs="
              f"{(ft.symbol_rate or 0)/1e3:.1f} kHz (snr {ft.symbol_rate_snr or 0:.1f})"
              f"  cv={ft.envelope_cv:.3f}  tones={ft.n_tones}"
              f"  fft={ft.ofdm_fft} cp={ft.ofdm_cp} scs="
              f"{(ft.subcarrier_spacing or 0)/1e3:.1f} kHz"
              f"  chirp={ft.chirp_score} sf={ft.css_sf}")
        print("  runners-up:", ", ".join(f"{c['name']} {c['score']*100:.0f}%"
                                        for c in r.candidates[1:]))
        if ok_mod and ok_name:
            passed += 1
        else:
            failed += 1
    print("=" * 78)
    print(f"passed {passed} / {passed + failed}")
    return 0 if failed == 0 else 1

A single realisation first, at a fixed seed, so the per-case measurements are
visible.

In [ ]:
RNG = np.random.default_rng(0xC0FFEE)      # reproducible
status = run_blind_validation()
print(f"\nexit status {status}")

Then the honest version: the same suite over independent noise realisations.

In [ ]:
def multiseed_validation(seeds):
    global RNG
    fails, total = {}, 0
    for seed in seeds:
        RNG = np.random.default_rng(seed)
        for label, fs, fc, iq, want_mod, want_name in cases():
            total += 1
            res = identify(iq, fs, rf_center_hz=fc, top=3)
            if not res:
                fails.setdefault(label, []).append((seed, "no burst", ""))
                continue
            r = max(res, key=lambda q: q.features.duration)
            ok = (r.features.modclass == want_mod
                  and (want_name is None
                       or (r.best and want_name.lower() in r.best["name"].lower())))
            if not ok:
                fails.setdefault(label, []).append(
                    (seed, r.features.modclass, (r.best or {}).get("name", "")[:26]))
    return total, fails

seeds = range(1, 5) if QUICK else range(1, 13)
total, fails = multiseed_validation(seeds)
nf = sum(len(v) for v in fails.values())
print(f"passed {total - nf} / {total}  ({len(list(seeds))} seeds x 8 cases)\n")
for k, v in sorted(fails.items(), key=lambda kv: -len(kv[1])):
    print(f"  {k}: {len(v)}/{len(list(seeds))} failed -> {v[:2]}")
if not fails:
    print("  no failures")
RNG = np.random.default_rng(0xC0FFEE)

## 19. Segmentation, and the blind SNR floor

The SNR sweep is the motivation for Part V. Classification is solid at 20–30 dB,
degrades through 15, and by 6 dB the modulation family is wrong. At 0 dB the
energy detector finds **no burst at all** — the deeper problem, because a
pipeline gated on burst detection cannot recover below that point however good
its features are.

The low-confidence flag fires correctly throughout the degraded region: the
classifier fails *honestly*, which is the minimum acceptable behaviour.

In [ ]:
def extra_checks():
    """Multi-burst segmentation and an SNR sweep."""
    print("\n" + "=" * 78)
    print("MULTI-BURST CAPTURE (3 protocols interleaved at fs=8 MHz)")
    fs = 8e6
    # one global noise floor; splicing separately-padded chunks would create
    # zero-power gaps and give the noise-floor estimator a false reference
    def gap(us):
        return np.zeros(int(us * 1e-6 * fs), np.complex64)

    cores = [
        (gap(300), None),
        (gen_gfsk(fs, 1e6, 250e3, 300), "BLE-ish"),
        (gap(400), None),
        (gen_ppm_ook(fs, 1e6, 120), "ADS-B-ish"),
        (gap(400), None),
        (gen_gfsk(fs, 250e3, 60e3, 400), "SiK-ish"),
        (gap(300), None),
    ]
    body = np.concatenate([c for c, _ in cores])
    ref = np.mean(np.abs(np.concatenate([c for c, n in cores if n])) ** 2)
    nf = np.sqrt(ref / (2 * 10 ** (25 / 10)))
    parts = [body + nf * (RNG.standard_normal(body.size)
                          + 1j * RNG.standard_normal(body.size))]
    iq = np.concatenate(parts).astype(np.complex64)
    res = identify(iq, fs, rf_center_hz=None, top=1)
    print(f"  {len(iq)} samples -> {len(res)} bursts")
    for i, r in enumerate(res):
        nm = r.best["name"] if r.best else "(none)"
        print(f"   {i}: t={r.features.t0*1e6:7.1f} us  {r.features.duration*1e6:7.1f} us  "
              f"{r.features.modclass:15s} {nm}"
              f"{'' if r.confident else '   [low confidence]'}")

    print("\n" + "=" * 78)
    print("SNR SWEEP — SiK-class 250 kbps GFSK at fs=2 MHz, fc=915 MHz")
    for snr in (30, 20, 15, 10, 6, 3, 0):
        iq = pad(gen_gfsk(2e6, 250e3, 60e3, 700), 2e6, 1e-3, 1e-3, snr_db=snr)
        res = identify(iq, 2e6, rf_center_hz=915e6, top=1)
        if not res:
            print(f"   {snr:3d} dB: no burst detected")
            continue
        r = max(res, key=lambda q: q.features.duration)
        nm = r.best["name"] if r.best else "(none)"
        print(f"   {snr:3d} dB: {r.features.modclass:12s} Rs="
              f"{(r.features.symbol_rate or 0)/1e3:7.1f} kHz  tones={r.features.n_tones}"
              f"  -> {nm[:40]}{'' if r.confident else ' [low conf]'}")

In [ ]:
extra_checks()

# Part V — The preamble matched filter

## 20. Why the floor exists

The 20 dB floor is structural. Every feature in Part II is a **second-order
statistic** — a bandwidth, an envelope variance, a histogram of a
nonlinearly-derived quantity. Two consequences:

1. Second-order statistics of a signal in noise converge at a rate no estimator
   design improves.
2. The frequency discriminator has a **threshold effect**. The small-noise
   approximation $\theta \approx \operatorname{Im}\{we^{-j\phi}\}/A$ underlying
   §9 requires $\sigma_\theta \ll 1$; below about 10 dB it fails and the
   discriminator produces clicks rather than a mildly noisy estimate.

A **matched filter** is a first-order statistic and is the provably optimal
detector for a *known* waveform in AWGN. The key observation is that the
waveform genuinely is known: most protocols open every frame with a fixed,
publicly documented preamble and sync word.

## 21. Detection theory

Correlate against a reference $r$ of $N$ samples, normalised so the statistic is
scale-free:

$$\rho[n] \;=\; \frac{\bigl|\sum_{k=0}^{N-1} x[n+k]\,r^*[k]\bigr|}
{\bigl\|x[n{:}n{+}N]\bigr\|\;\bigl\|r\bigr\|} \;\in\; [0, 1]$$

### Noise only

For circular complex Gaussian noise, $\rho^2 \sim \mathrm{Beta}(1, N-1)$, so

$$\boxed{\;P(\rho > t) \;=\; \bigl(1 - t^2\bigr)^{N-1} \;\approx\; e^{-(N-1)t^2}\;} \tag{1}$$

The false-alarm probability falls **exponentially in template length**. This is
the entire mechanism: the threshold needed for a fixed $P_{fa}$ shrinks as
$1/\sqrt{N}$.

### Signal present

With a matched, constant-amplitude reference, write $x = Ar + w$. The numerator
accumulates coherently, $\sum x_k r_k^* = AN + \sum w_k r_k^*$, where the second
term is complex Gaussian of variance $N\sigma^2$ — growing as $\sqrt{N}$ against
the signal's $N$. The denominator is $\sqrt{N(A^2+\sigma^2)}\cdot\sqrt{N}$. Hence

$$\boxed{\;\mathbb{E}[\rho] \;\approx\; \sqrt{\frac{\gamma}{1+\gamma}}\;} \tag{2}$$

**Note what (2) does not contain: $N$.** The peak correlation is set purely by
SNR and saturates at 1. Longer templates do not raise the peak — all of their
benefit lives in (1), in the threshold we are permitted to use. This is a
genuinely counter-intuitive point and §25 verifies it directly across a 21×
range of template lengths.

### Threshold and floor

With $M$ effective trials (positions × frequency bins × templates), invert (1):

$$\boxed{\;t \;=\; \sqrt{\frac{\ln(M/P_{fa})}{N-1}}\;} \tag{3}$$

Setting (2) above (3) and using $\gamma/(1+\gamma)\approx\gamma$ for small
$\gamma$:

$$\boxed{\;\gamma_{\min} \;\approx\; \frac{\ln(M/P_{fa})}{N}\;} \tag{4}$$

The processing gain is $N/\ln(M/P_{fa})$ — template length, discounted by the
logarithm of the search space. Searching harder costs remarkably little.

| Template | $N$ | threshold $t$ | $\gamma_{\min}$ |
|---|---|---|---|
| SiK, 28 bits @ 8 sps | 224 | 0.32 | −9.4 dB |
| BLE advertising, 40 bits @ 8 sps | 320 | 0.27 | −11.1 dB |
| POCSAG, 128 bits @ 27 sps (after 2x decimation) | 3413 | 0.08 | −21.7 dB |
| LoRa SF7, 8 upchirps @ 4 sps | 4096 | 0.08 | −22.5 dB |

This also explains something about LoRa that is otherwise mysterious: its
ability to demodulate below the noise floor is not exotic, it is equation (4)
with a very long preamble.

In [ ]:
def threshold_for_pfa(n_template: int, n_trials: int, p_fa: float = 1e-6) -> float:
    """Threshold from equation (3). Never returns more than 0.999."""
    n_eff = max(2, n_template) - 1
    t = math.sqrt(max(0.0, math.log(max(n_trials, 1) / max(p_fa, 1e-300))) / n_eff)
    return float(min(t, 0.999))


def snr_floor_db(n_template: int, n_trials: int, p_fa: float = 1e-6) -> float:
    """Per-sample SNR floor from equation (4), in dB."""
    g = math.log(max(n_trials, 1) / max(p_fa, 1e-300)) / max(n_template, 1)
    g = min(g, 0.999)                     # gamma/(1+gamma) cannot exceed 1
    return 10.0 * math.log10(g / max(1.0 - g, 1e-12))

## 22. Reference waveform synthesis and the template bank

All synthesis supports **fractional** samples-per-symbol, because the capture
rate is whatever the SDR provided and is rarely an integer multiple of the
protocol's symbol rate.

### How much of each template is actually determined by its standard

This distinction matters more than any code here, so it is recorded per template
in a `verified` flag:

- **BLE advertising** — fully determined. The 1M preamble is `0xAA` and the
  advertising access address is the fixed constant `0x8E89BED6`; whitening
  begins *after* the access address, so all 40 bits are deterministic. Bit
  order is the subtlety: octets transmit LSB-first and the address goes out
  least-significant-octet first.
- **ADS-B 1090ES** — fully determined. Four 0.5 µs pulses at 0, 1.0, 3.5 and
  4.5 µs.
- **LoRa** — structurally determined (8 base upchirps) even though the PHY
  itself is Semtech IP.
- **SiK/MAVLink** — flagged `verified=False`. `0x2DD4` is the Si4432 silicon
  default and what stock builds use, but both sync word and preamble length are
  register-configurable, so a modified deployment will not match. The flag is
  about configurability, not doubt about the constant.

### One template deliberately omitted

Zigbee/802.15.4 would be a good candidate — the O-QPSK symbol-to-chip mapping is
public and the 32-chip sequence for symbol zero would make a strong template. I
left it out because **I do not trust my recall of that sequence**, and the
failure mode is nasty: since the test generator would use the same constant, a
wrong sequence would correlate perfectly with itself and *pass its own test
suite* while never matching a real capture. Self-consistent tests cannot detect
a wrong constant. Adding it requires reading the table out of IEEE 802.15.4
directly.

The same caution applies to everything cited here — these are constants recalled
from specifications, not read out of them, and the BLE bit order in particular is
worth verifying against the document before trusting a real capture.

In [ ]:
def _bits(spec) -> np.ndarray:
    """Accept a bit list, a '1010' string, or ('hex', '2DD4', nbits)."""
    if isinstance(spec, str):
        return np.array([int(c) for c in spec if c in "01"], dtype=np.int8)
    if isinstance(spec, tuple) and spec and spec[0] == "hex":
        val, nbits = int(spec[1], 16), spec[2]
        return np.array([(val >> (nbits - 1 - i)) & 1 for i in range(nbits)],
                        dtype=np.int8)
    return np.asarray(spec, dtype=np.int8)


def synth_gfsk(bits: np.ndarray, fs: float, symbol_rate: float,
               deviation: float, bt: float = 0.5) -> np.ndarray:
    """Gaussian-filtered 2-FSK reference.

    The Gaussian pulse follows the usual definition with
    ``sigma = sqrt(ln 2) / (2*pi*BT)`` in symbol units. ``deviation`` is the
    *peak* deviation, so the modulation index is ``h = 2*deviation/symbol_rate``
    (h = 0.5 for BLE, giving 250 kHz at 1 Msym/s).
    """
    sps = fs / symbol_rate
    n = int(round(bits.size * sps))
    idx = np.clip((np.arange(n) / sps).astype(int), 0, bits.size - 1)
    nrz = 2.0 * bits[idx] - 1.0

    sigma = math.sqrt(math.log(2.0)) / (2.0 * math.pi * bt) * sps
    half = max(1, int(math.ceil(3.0 * sigma)))
    t = np.arange(-half, half + 1)
    g = np.exp(-0.5 * (t / max(sigma, 1e-9)) ** 2)
    g /= g.sum()
    shaped = np.convolve(nrz, g, mode="same")

    phase = 2.0 * np.pi * deviation * np.cumsum(shaped) / fs
    return np.exp(1j * phase).astype(np.complex64)


def synth_css_upchirps(fs: float, bw: float, sf: int, n_chirps: int) -> np.ndarray:
    """``n_chirps`` back-to-back base (symbol-zero) LoRa upchirps.

    A LoRa symbol sweeps the full bandwidth once, so the chirp rate is
    ``k = bw / T_sym`` with ``T_sym = 2^sf / bw``, i.e. ``k = bw^2 / 2^sf``.
    """
    n_sym = int(round(2 ** sf * fs / bw))
    t = np.arange(n_sym) / fs
    k = bw * bw / (2 ** sf)
    f = -bw / 2.0 + k * t
    one = np.exp(1j * 2.0 * np.pi * np.cumsum(f) / fs)
    return np.tile(one, n_chirps).astype(np.complex64)


def synth_ook_pulses(fs: float, pulses: Sequence[tuple[float, float]],
                     total_s: float, rise_s: float = 0.05e-6) -> np.ndarray:
    """On-off keyed reference from (start_seconds, width_seconds) pulse list.

    Returned real-valued. Correlating a real reference against complex IQ is
    correct here: the ``|.|`` in rho makes the statistic invariant to the
    unknown constant carrier phase.
    """
    n = int(round(total_s * fs))
    x = np.zeros(n, dtype=np.float64)
    for start, width in pulses:
        a, b = int(round(start * fs)), int(round((start + width) * fs))
        x[max(0, a):max(0, b)] = 1.0
    w = max(2, int(round(rise_s * fs)))
    x = np.convolve(x, np.ones(w) / w, mode="same")
    return x.astype(np.complex64)

In [ ]:
@dataclass
class Template:
    name: str                       # matches a PROTOCOLS entry in iq_protocol_id
    kind: str                       # 'gfsk' | 'css' | 'ook'
    symbol_rate: float
    params: dict
    band: list[tuple[float, float]]
    spec: str
    verified: bool = True
    bits: Optional[str] = None
    cfo_span: Optional[float] = None   # +/- Hz to search; default 0.3*symbol_rate

    def duration(self) -> float:
        if self.kind == "ook":
            return float(self.params["total_s"])
        if self.kind == "css":
            p = self.params
            return p["n_chirps"] * (2 ** p["sf"]) / p["bw"]
        return len(_bits(self.bits)) / self.symbol_rate

    def synth(self, fs: float) -> np.ndarray:
        if self.kind == "gfsk":
            return synth_gfsk(_bits(self.bits), fs, self.symbol_rate,
                              self.params["deviation"], self.params.get("bt", 0.5))
        if self.kind == "css":
            p = self.params
            return synth_css_upchirps(fs, p["bw"], p["sf"], p["n_chirps"])
        if self.kind == "ook":
            return synth_ook_pulses(fs, self.params["pulses"],
                                    self.params["total_s"])
        raise ValueError(f"unknown template kind {self.kind!r}")

In [ ]:
MHz, kHz = 1e6, 1e3


_BLE_AA = "01101011011111011001000101110001"   # 0x8E89BED6, LSB-first per octet


_BLE_PREAMBLE_1M = "01010101"                  # 0xAA


BLE_ADV_1M = Template(
    name="Bluetooth LE 1M (GFSK)",
    kind="gfsk", symbol_rate=1e6,
    params={"deviation": 250 * kHz, "bt": 0.5},
    bits=_BLE_PREAMBLE_1M + _BLE_AA,
    band=[(2.402e9, 2.480e9)],
    spec="Bluetooth Core 5.4, Vol 6 Part B 2.1.2 (access address 0x8E89BED6)",
    cfo_span=250 * kHz,
)


BLE_ADV_2M = Template(
    name="Bluetooth LE 2M (GFSK)",
    kind="gfsk", symbol_rate=2e6,
    params={"deviation": 500 * kHz, "bt": 0.5},
    bits="01010101" * 2 + _BLE_AA,             # 2M PHY uses a 16-bit preamble
    band=[(2.402e9, 2.480e9)],
    spec="Bluetooth Core 5.4, Vol 6 Part B 2.1.2",
    cfo_span=400 * kHz,
)


ADSB_PREAMBLE = Template(
    name="ADS-B 1090ES (PPM/OOK)",
    kind="ook", symbol_rate=1e6,
    params={"pulses": [(0.0, 0.5e-6), (1.0e-6, 0.5e-6),
                       (3.5e-6, 0.5e-6), (4.5e-6, 0.5e-6)],
            "total_s": 8.0e-6},
    band=[(1089e6, 1091e6)],
    spec="RTCA DO-260B 2.2.3.2.1.1 / ICAO Annex 10 Vol IV",
    cfo_span=0.0,                              # OOK envelope: no carrier phase
)


def lora_preamble(bw: float = 125 * kHz, sf: int = 7, n_chirps: int = 8) -> Template:
    return Template(
        name="LoRa / CSS (LoRaWAN, Meshtastic, ELRS)",
        kind="css", symbol_rate=bw / (2 ** sf),
        params={"bw": bw, "sf": sf, "n_chirps": n_chirps},
        band=[(150e6, 960e6), (2.400e9, 2.485e9)],
        spec="LoRa Alliance regional params; preamble = 8 base upchirps",
        cfo_span=0.25 * bw,
    )


def sik_preamble(symbol_rate: float = 250e3, deviation: float = 60 * kHz,
                 lead_bits: int = 12, sync_hex: str = "2DD4") -> Template:
    return Template(
        name="SiK / MAVLink telemetry (2-FSK/GFSK)",
        kind="gfsk", symbol_rate=symbol_rate,
        params={"deviation": deviation, "bt": 0.5},
        bits="10" * (lead_bits // 2) + "".join(
            f"{int(sync_hex, 16):0{len(sync_hex) * 4}b}"),
        band=[(410e6, 480e6), (863e6, 928e6), (2.400e9, 2.485e9)],
        spec="SiK firmware / Si4432 datasheet (sync word is configurable)",
        verified=False,
        cfo_span=0.3 * symbol_rate,
    )


def pocsag_preamble(symbol_rate: float = 1200.0, deviation: float = 4.5 * kHz,
                    n_bits: int = 128) -> Template:
    return Template(
        name="POCSAG paging (2-FSK)",
        kind="gfsk", symbol_rate=symbol_rate,
        params={"deviation": deviation, "bt": 1.0},
        bits="10" * (n_bits // 2),
        band=[(130e6, 960e6)],
        spec="ITU-R M.584-2 (576-bit alternating preamble)",
        cfo_span=2.0 * symbol_rate,
    )


def default_templates() -> list[Template]:
    return [
        BLE_ADV_1M,
        BLE_ADV_2M,
        ADSB_PREAMBLE,
        lora_preamble(125 * kHz, 7),
        lora_preamble(125 * kHz, 9),
        lora_preamble(250 * kHz, 7),
        sik_preamble(250e3, 60 * kHz),
        sik_preamble(64e3, 30 * kHz),
        pocsag_preamble(1200.0),
    ]

In [ ]:
# verify the BLE access-address bit order by reconstructing it
def lsb_first(byte):
    return "".join(str((byte >> i) & 1) for i in range(8))

aa = 0x8E89BED6
octets = [(aa >> s) & 0xFF for s in (0, 8, 16, 24)]     # LSO first: D6 BE 89 8E
rebuilt = "".join(lsb_first(o) for o in octets)
print(f"access address 0x{aa:08X} -> octets LSO first: {[f'{o:02X}' for o in octets]}")
print(f"rebuilt : {rebuilt}")
print(f"in module: {_BLE_AA}")
print(f"match: {rebuilt == _BLE_AA}")

## 23. The correlator, and a nearly free frequency search

### Carrier offset is the main fragility

Coherent correlation is why this works and also what breaks it. A residual
offset $\Delta f$ rotates the reference through $\Delta f \cdot T$ cycles over
the template duration, and the coherent sum degrades as $|\mathrm{sinc}(\Delta f T)|$
— one full cycle destroys the gain completely. At 915 MHz a 10 ppm crystal is
9 kHz, and over a 112 µs SiK preamble that *is* one full cycle. So we must
search a frequency grid, spaced at $0.8/T$ to hold worst-case scalloping loss
near 1 dB.

### The grid is almost free

De-rotating the signal by $-\Delta f$ is equivalent to rotating the reference by
$+\Delta f$. Writing the correlation as a convolution with
$h[j] = r^*[N-1-j]$, the rotated kernel is

$$h_{\Delta f}[j] = r^*[N{-}1{-}j]\,e^{-j2\pi\Delta f (N-1-j)/f_s}
= \underbrace{e^{-j2\pi \Delta f (N-1)/f_s}}_{\text{constant phase}}\;
h[j]\,e^{+j2\pi\Delta f j/f_s}$$

Multiplying a sequence by that exponential **shifts its DFT** by
$k = \Delta f \cdot N_{\text{FFT}}/f_s$ bins, and the leading constant phase dies
inside the $|\cdot|$. So we transform the template **once** and circularly shift
its spectrum per frequency bin: one inverse FFT per bin instead of a fresh
correlation.

The shift direction is easy to get backwards — an earlier version used
`roll(H, -k)` and reported every offset with inverted sign while detecting
perfectly, since a symmetric grid hides the error in the peak. §25 checks the
sign against injected offsets for exactly this reason.

In [ ]:
def _next_fast_len(n: int) -> int:
    return int(sp_fft.next_fast_len(n))


def normalized_correlate(x: np.ndarray, r: np.ndarray,
                         cfo_hz: Sequence[float] = (0.0,),
                         fs: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Normalised correlation of ``x`` against ``r`` over a frequency grid.

    Returns ``(rho, best_cfo)``, both length ``len(x) - len(r) + 1``, where
    ``rho[n]`` is the best normalised correlation at offset ``n`` across the
    grid and ``best_cfo[n]`` is the grid point that achieved it.

    The frequency search reuses one template transform. Rotating the reference
    by ``+df`` shifts its spectrum; because ``rho`` takes a magnitude, the
    constant phase that shift introduces cancels, so each grid point costs one
    inverse FFT instead of a fresh correlation.
    """
    x = np.asarray(x, dtype=np.complex128)
    r = np.asarray(r, dtype=np.complex128)
    n, m = x.size, r.size
    if n < m:
        return np.zeros(0), np.zeros(0)

    out_len = n - m + 1
    nfft = _next_fast_len(n + m)

    # sliding energy of x over an m-sample window
    p = np.abs(x) ** 2
    cs = np.concatenate(([0.0], np.cumsum(p)))
    e_x = cs[m:] - cs[:-m]                       # length out_len
    e_r = float(np.sum(np.abs(r) ** 2))
    denom = np.sqrt(np.maximum(e_x, 1e-30) * max(e_r, 1e-30))

    X = np.fft.fft(x, nfft)
    h = np.conj(r[::-1])                         # correlation as convolution
    H = np.fft.fft(h, nfft)

    # frequency grid quantised to FFT bins so the shift is exact
    bin_hz = fs / nfft
    shifts = sorted({int(round(f / bin_hz)) for f in cfo_hz})

    # h_f[j] = C * h[j] * exp(+j2*pi*f*j/fs), and multiplying a sequence by that
    # exponential shifts its DFT *up* by k = f*nfft/fs bins. The leftover
    # constant C is a pure phase and dies in the |.|.
    rho = np.zeros(out_len)
    best = np.zeros(out_len)
    for k in shifts:
        c = np.fft.ifft(X * np.roll(H, k))[m - 1 : m - 1 + out_len]
        cand = np.abs(c) / denom
        upd = cand > rho
        rho[upd] = cand[upd]
        best[upd] = k * bin_hz
    return rho, best


def _cfo_grid(span_hz: float, duration_s: float) -> np.ndarray:
    """Grid spaced at 0.8/T, which holds worst-case scalloping under ~1 dB."""
    if span_hz <= 0 or duration_s <= 0:
        return np.array([0.0])
    step = 0.8 / duration_s
    k = int(math.ceil(span_hz / step))
    return np.arange(-k, k + 1) * step

## 24. The bank, and template cross-talk

### A limit of equation (1)

Equation (1) bounds false alarms against **noise**. It says nothing about one
template correlating with a *different real signal*, and several of these
templates genuinely resemble each other — BLE 1M, BLE 2M and SiK all open with
an alternating GFSK pattern, so a BLE frame lights up all three. That is
cross-talk, not a false alarm, and no threshold fixes it because the correlation
really is present.

Two mechanisms handle it:

**Non-maximum suppression in time.** A frame is one protocol, so among hits
whose template spans overlap, only the largest margin $\rho/t$ survives.
Suppressed hits are retained on `.shadowed`. *Limitation:* two genuinely
concurrent signals on different channels overlap in time and the weaker is lost
— separate by frequency first if that case matters.

**A consistency gate, from equation (2).** A truly matched template must return
$\rho \approx \sqrt{\gamma/(1+\gamma)}$, so its *implied* SNR should track the
SNR measured on the burst. A template correlating with a frame's random payload
clears the noise-only threshold but implies far less signal than is present —
this is how a BLE 2M template found a "preamble" inside a 30 dB BLE 1M payload,
reporting an implied −2.0 dB inside a +31.3 dB burst.

The gate carries its own trap, and it is instructive. Applied unconditionally it
**cost 12 dB of POCSAG sensitivity**: near the energy detector's floor the
measured burst SNR is whatever a noise fluctuation happened to do, and a spurious
+2.7 dB burst vetoed a perfectly good correlation at −12 dB. It is now applied
only to bursts above 10 dB, where the measurement means something, and is
one-sided — a hit is only ever rejected for implying *less* signal than is
measurably present.

### The correlator runs on the raw capture

Deliberately not on detected bursts. Below roughly 6 dB the energy detector
finds nothing at all, so requiring it to fire first would reintroduce the exact
floor we are removing.

### Channel filtering buys no sensitivity here

Worth stating precisely, because it contradicts §8. In the language of (4),
decimating by $D$ raises $\gamma$ by $\approx D$ and shrinks $N$ by the same
factor, leaving $\gamma N$ — and therefore the floor — invariant. The matched
filter already *is* the optimal filter. Decimation is retained purely for
compute.

In [ ]:
@dataclass
class PreambleHit:
    name: str
    rho: float
    threshold: float
    t0: float                    # seconds into the capture
    sample: int
    cfo_hz: float
    n_template: int
    template_duration: float
    processing_gain_db: float
    est_snr_db: float
    spec: str
    verified: bool = True
    decimation: int = 1
    shadowed: list = field(default_factory=list)   # cross-talk, suppressed

    @property
    def margin(self) -> float:
        return self.rho / max(self.threshold, 1e-9)

    def __str__(self) -> str:
        flag = "" if self.verified else "  [template constants configurable — verify]"
        shad = (f", shadowed {len(self.shadowed)}" if self.shadowed else "")
        return (f"{self.name}: rho={self.rho:.3f} vs threshold {self.threshold:.3f} "
                f"(margin {self.margin:.2f}x) at t={self.t0*1e6:.1f} us, "
                f"CFO={self.cfo_hz/1e3:+.1f} kHz, N={self.n_template}, "
                f"gain={self.processing_gain_db:.1f} dB, "
                f"est SNR={self.est_snr_db:+.1f} dB{shad}{flag}")

In [ ]:
class PreambleBank:
    """Matched-filter bank over documented preambles and sync words."""

    def __init__(self, templates: Optional[Sequence[Template]] = None,
                 p_fa: float = 1e-6, sps_target: float = 8.0,
                 max_block: int = 1 << 18, min_margin: float = 1.0,
                 suppress_overlap: bool = True):
        self.templates = list(templates) if templates is not None else default_templates()
        self.p_fa = p_fa
        self.sps_target = sps_target
        self.max_block = max_block
        self.min_margin = min_margin
        self.suppress_overlap = suppress_overlap

    # -- helpers ------------------------------------------------------------
    def _bandwidth_of(self, tpl: Template) -> float:
        """Rough occupied bandwidth, used only to choose a decimation factor."""
        if tpl.kind == "css":
            return float(tpl.params["bw"])
        if tpl.kind == "ook":
            return 4.0 * tpl.symbol_rate
        dev = float(tpl.params["deviation"])
        return 2.0 * (dev + tpl.symbol_rate / 2.0)      # Carson

    def _decimate(self, x: np.ndarray, fs: float, tpl: Template):
        """Channel-filter and decimate for speed only.

        This buys no sensitivity — the matched filter is already the optimal
        filter, and in the language of equation (4) decimating by D raises
        gamma by ~D while shrinking N by the same factor. It buys compute.
        """
        bw = self._bandwidth_of(tpl)
        target = max(self.sps_target * tpl.symbol_rate, 2.5 * bw)
        d = int(max(1, math.floor(fs / target)))
        if d < 2 or x.size // d < 256:
            return x, fs, 1
        taps = min(193, (x.size // 4) | 1)
        if taps < 9:
            return x, fs, 1
        h = sps_signal.firwin(taps, 1.0 / d, window=("kaiser", 6.0))
        y = sps_signal.lfilter(h, 1.0, x)[::d].astype(np.complex64)
        return y, fs / d, d

    # -- main ---------------------------------------------------------------
    def scan(self, x: np.ndarray, fs: float,
             rf_center_hz: Optional[float] = None,
             derotate_hz: float = 0.0) -> list[PreambleHit]:
        """Correlate ``x`` against every in-band template.

        Runs on the *raw capture*, deliberately not on energy-detected bursts:
        below roughly 6 dB the energy detector finds nothing at all, and
        requiring it to fire first would reintroduce the floor we are removing.
        """
        x = np.asarray(x, dtype=np.complex64)
        if derotate_hz:
            n = np.arange(x.size)
            x = (x * np.exp(-2j * np.pi * derotate_hz * n / fs)).astype(np.complex64)

        hits: list[PreambleHit] = []
        for tpl in self.templates:
            if rf_center_hz is not None and not any(
                    lo <= rf_center_hz <= hi for lo, hi in tpl.band):
                continue

            xd, fsd, decim = self._decimate(x, fs, tpl)
            r = tpl.synth(fsd)
            if r.size < 16 or r.size > xd.size:
                continue

            span = tpl.cfo_span if tpl.cfo_span is not None else 0.3 * tpl.symbol_rate
            grid = _cfo_grid(span, tpl.duration())

            # trials: independent positions x frequency bins. Adjacent samples
            # are correlated over a template length, so the effective number of
            # independent positions is the record length in template durations,
            # inflated by 4 for safety.
            n_pos = max(1, int(4 * xd.size / r.size))
            n_trials = n_pos * grid.size * max(1, len(self.templates))
            thr = threshold_for_pfa(r.size, n_trials, self.p_fa)

            rho, cfo = self._scan_blocks(xd, r, grid, fsd)
            if rho.size == 0:
                continue
            k = int(np.argmax(rho))
            if rho[k] < thr * self.min_margin:
                continue

            # invert equation (2) for a post-detection SNR estimate
            p = min(float(rho[k]) ** 2, 0.999999)
            gamma = p / (1.0 - p)
            hits.append(PreambleHit(
                name=tpl.name, rho=float(rho[k]), threshold=thr,
                t0=k / fsd, sample=k * decim, cfo_hz=float(cfo[k]),
                n_template=int(r.size), template_duration=tpl.duration(),
                processing_gain_db=10.0 * math.log10(
                    r.size / math.log(max(n_trials, 2) / self.p_fa)),
                est_snr_db=10.0 * math.log10(max(gamma, 1e-12)),
                spec=tpl.spec, verified=tpl.verified, decimation=decim,
            ))

        hits.sort(key=lambda h: -(h.rho / max(h.threshold, 1e-9)))
        return self._suppress(hits) if self.suppress_overlap else hits

    @staticmethod
    def _suppress(hits: list[PreambleHit]) -> list[PreambleHit]:
        """Non-maximum suppression in time.

        Equation (1) bounds the false-alarm rate against *noise*. It says
        nothing about one template correlating with a different real signal, and
        several of these templates genuinely resemble each other — BLE 1M, BLE
        2M and SiK all open with an alternating GFSK pattern, so a BLE frame
        will light up all three. That is template cross-talk, not a false alarm,
        and no threshold can fix it because the correlation really is there.

        What does fix it: a frame is one protocol, so among hits whose template
        spans overlap in time only the largest margin survives. The suppressed
        hits are returned on ``.shadowed`` for inspection.

        Limitation: two genuinely concurrent signals on different channels
        within one capture will overlap in time, and the weaker one is lost.
        Pass ``suppress_overlap=False`` and separate by frequency first if that
        case matters.
        """
        kept: list[PreambleHit] = []
        for h in hits:                       # already sorted by descending margin
            a0, a1 = h.t0, h.t0 + h.template_duration
            clash = next((k for k in kept
                          if a0 < k.t0 + k.template_duration and k.t0 < a1), None)
            if clash is None:
                kept.append(h)
            else:
                clash.shadowed.append(h)
        return kept

    def _scan_blocks(self, x: np.ndarray, r: np.ndarray,
                     grid: np.ndarray, fs: float):
        """Overlap-save so memory stays bounded on long captures."""
        m = r.size
        if x.size <= self.max_block:
            return normalized_correlate(x, r, grid, fs)

        step = self.max_block - (m - 1)
        rho_parts, cfo_parts = [], []
        for start in range(0, x.size - m + 1, step):
            blk = x[start : start + step + m - 1]
            if blk.size < m:
                break
            a, b = normalized_correlate(blk, r, grid, fs)
            rho_parts.append(a)
            cfo_parts.append(b)
        if not rho_parts:
            return np.zeros(0), np.zeros(0)
        return np.concatenate(rho_parts), np.concatenate(cfo_parts)

## 25. Validation

Four checks, then the headline measurement.

In [ ]:
def awgn_at(x: np.ndarray, snr_db: float) -> np.ndarray:
    """Add noise scaled to the *signal* power, so SNR is per-sample in-band."""
    p = float(np.mean(np.abs(x) ** 2))
    n = math.sqrt(p / (2 * 10 ** (snr_db / 10)))
    return (x + n * (RNG.standard_normal(x.size)
                     + 1j * RNG.standard_normal(x.size))).astype(np.complex64)


def frame_gfsk(tpl: Template, fs: float, payload_bits: int,
               snr_db: float, lead_s: float, cfo_hz: float = 0.0) -> np.ndarray:
    """Preamble + random payload, embedded in a longer noise-only record."""
    bits = np.concatenate([_bits(tpl.bits),
                           RNG.integers(0, 2, payload_bits).astype(np.int8)])
    core = synth_gfsk(bits, fs, tpl.symbol_rate,
                      tpl.params["deviation"], tpl.params.get("bt", 0.5))
    if cfo_hz:
        core = core * np.exp(2j * np.pi * cfo_hz * np.arange(core.size) / fs)

    lead = int(round(lead_s * fs))
    tail = lead
    p = float(np.mean(np.abs(core) ** 2))
    nf = math.sqrt(p / (2 * 10 ** (snr_db / 10)))
    noise = lambda n: nf * (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
    return np.concatenate([noise(lead), awgn_at(core, snr_db),
                           noise(tail)]).astype(np.complex64)


def frame_lora(fs: float, bw: float, sf: int, n_data: int,
               snr_db: float, lead_s: float) -> np.ndarray:
    pre = synth_css_upchirps(fs, bw, sf, 8)
    n = int(round(2 ** sf * fs / bw))
    t = np.arange(n) / fs
    k = bw * bw / (2 ** sf)
    data = []
    for _ in range(n_data):
        off = RNG.integers(0, 2 ** sf) / 2 ** sf
        f = -bw / 2 + bw * ((t / (n / fs) + off) % 1.0)
        data.append(np.exp(1j * 2 * np.pi * np.cumsum(f) / fs))
    core = np.concatenate([pre] + data).astype(np.complex64)
    lead = int(round(lead_s * fs))
    p = float(np.mean(np.abs(core) ** 2))
    nf = math.sqrt(p / (2 * 10 ** (snr_db / 10)))
    noise = lambda m: nf * (RNG.standard_normal(m) + 1j * RNG.standard_normal(m))
    return np.concatenate([noise(lead), awgn_at(core, snr_db),
                           noise(lead)]).astype(np.complex64)

In [ ]:
def test_correlator_matches_brute_force():
    print("=" * 78)
    print("1. FFT/spectral-shift correlator vs direct evaluation")
    fs = 8e6
    r = BLE_ADV_1M.synth(fs)
    grid = _cfo_grid(BLE_ADV_1M.cfo_span, BLE_ADV_1M.duration())
    x = (RNG.standard_normal(1500) + 1j * RNG.standard_normal(1500)) * 0.3
    x = x.astype(np.complex64)
    x[400:400 + r.size] += r * np.exp(2j * np.pi * 37e3 * np.arange(r.size) / fs)

    nfft = sp_fft.next_fast_len(x.size + r.size)
    gq = sorted({round(f / (fs / nfft)) * (fs / nfft) for f in grid})

    er = np.sum(np.abs(r) ** 2)
    ref = np.zeros(x.size - r.size + 1)
    for i in range(ref.size):
        seg = x[i:i + r.size]
        ex = np.sum(np.abs(seg) ** 2)
        best = 0.0
        for f in gq:
            rr = r * np.exp(2j * np.pi * f * np.arange(r.size) / fs)
            best = max(best, abs(np.vdot(rr, seg)) / math.sqrt(ex * er))
        ref[i] = best

    rho, cfo = normalized_correlate(x, r, grid, fs)
    err = float(np.max(np.abs(rho - ref)))
    k = int(np.argmax(rho))
    print(f"   N={r.size}, {grid.size} frequency bins spaced "
          f"{grid[1]-grid[0]:.0f} Hz")
    print(f"   max |rho_fft - rho_direct| = {err:.2e}")
    print(f"   peak at sample {k} (true 400), CFO {cfo[k]/1e3:+.1f} kHz (true +37.0)")
    ok = err < 1e-6 and k == 400 and abs(cfo[k] - 37e3) < 25e3
    print(f"   -> {'OK' if ok else 'FAIL'}")
    return ok

In [ ]:
_ = test_correlator_matches_brute_force()

### Equation (2) and its independence of $N$

The claim to test is the surprising one: a template of 320 samples and one of
6827 samples should track the *same* $\rho$ curve. If they do, the processing
gain really does live entirely in the threshold.

In [ ]:
def test_rho_vs_snr():
    print("=" * 78)
    print("2. Peak correlation vs SNR — equation (2), and its independence of N")
    fs = 8e6
    templates = {"BLE 1M (N=320)": BLE_ADV_1M.synth(fs),
                 "POCSAG (N=%d)" % pocsag_preamble().synth(64e3).size:
                     pocsag_preamble().synth(64e3)}
    print(f"   {'SNR':>6}  {'theory':>8}   " +
          "   ".join(f"{k:>16}" for k in templates))
    ok = True
    for snr in (20, 10, 0, -6, -10, -15):
        g = 10 ** (snr / 10)
        theory = math.sqrt(g / (1 + g))
        row = []
        for r in templates.values():
            acc = []
            for _ in range(16):
                nf = math.sqrt(1 / (2 * g))
                y = r + nf * (RNG.standard_normal(r.size)
                              + 1j * RNG.standard_normal(r.size))
                acc.append(abs(np.vdot(r, y))
                           / (np.linalg.norm(r) * np.linalg.norm(y)))
            m = float(np.mean(acc))
            row.append(m)
            if abs(m - theory) > 0.05:
                ok = False
        print(f"   {snr:4d}dB  {theory:8.4f}   " +
              "   ".join(f"{v:16.4f}" for v in row))
    print(f"   -> {'OK' if ok else 'FAIL'} (both template lengths track the same "
          f"curve; N affects the threshold, not the peak)")
    return ok

In [ ]:
_ = test_rho_vs_snr()

### Sign of the frequency estimate

Detection is insensitive to the shift direction because the grid is symmetric,
so this needs an explicit test against injected offsets. It also confirms the
$0.8/T$ grid spacing holds the scalloping loss to a fraction of a dB across the
full search span.

In [ ]:
fs = 8e6
r = BLE_ADV_1M.synth(fs)
grid = _cfo_grid(BLE_ADV_1M.cfo_span, BLE_ADV_1M.duration())
print(f"grid: {grid.size} bins spaced {grid[1]-grid[0]:.0f} Hz "
      f"(0.8/T with T={BLE_ADV_1M.duration()*1e6:.0f} us)")
rhos = []
for true in (-120e3, -80e3, -37e3, 0.0, 37e3, 80e3, 120e3):
    x = ((RNG.standard_normal(1500) + 1j * RNG.standard_normal(1500)) * 0.3).astype(np.complex64)
    x[400:400 + r.size] += r * np.exp(2j * np.pi * true * np.arange(r.size) / fs)
    rho, cfo = normalized_correlate(x, r, grid, fs)
    k = int(np.argmax(rho)); rhos.append(rho[k])
    print(f"  injected {true/1e3:+7.1f} kHz -> reported {cfo[k]/1e3:+7.1f} kHz  "
          f"(sample {k}, rho={rho[k]:.4f})")
print(f"\nscalloping loss across the span: "
      f"{20*math.log10(max(rhos)/min(rhos)):.2f} dB")

### False alarms

Equation (1) predicts essentially none. Verifying on noise-only captures across
three bands checks that the threshold derivation is right, and that the
normalisation really is scale-free.

One caveat on interpretation. The trial count $M$ fed to equation (3) is a
*heuristic*: adjacent correlation samples are dependent over a template length,
so the number of independent positions is estimated as the record length in
template durations, inflated by 4 for safety. That makes the realised $P_{fa}$
approximate rather than exactly the nominal $10^{-6}$. Over 80 independent
noise scans I measured zero hits, but a different seed produced one — consistent
with a rate somewhat above nominal. If you need a calibrated false-alarm rate,
measure it empirically on your own noise rather than trusting the analytic
value.

In [ ]:
def test_false_alarms():
    print("=" * 78)
    print("3. False-alarm rate on noise-only captures")
    bank = PreambleBank(p_fa=1e-6)
    total = 0
    trials = 12
    for band, fs, n in ((2.44e9, 8e6, 200_000),
                        (915e6, 2e6, 200_000),
                        (1090e6, 8e6, 200_000)):
        fired = 0
        for _ in range(trials):
            x = (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
            hits = bank.scan(x.astype(np.complex64), fs, rf_center_hz=band)
            fired += len(hits)
        total += fired
        print(f"   fc={band/1e6:7.1f} MHz  fs={fs/1e6:g} MHz  "
              f"{trials} x {n} samples -> {fired} hits")
    print(f"   -> {'OK' if total == 0 else 'FAIL'} ({total} total false alarms)")
    return total == 0

In [ ]:
RNG = np.random.default_rng(0xBEEF)        # reproducible
_ = test_false_alarms()

### Carrier-offset robustness

Detection should hold across the whole grid span at 0 dB SNR and fail outside
it. The failure outside the span is expected behaviour, not a defect — it marks
where `cfo_span` needs widening for a given oscillator tolerance.

In [ ]:
def _blind_ok(iq, fs, fc, want) -> bool:
    res = identify(iq, fs, rf_center_hz=fc, top=1)
    if not res:
        return False
    r = max(res, key=lambda q: q.features.duration)
    return bool(r.confident and r.best and want.lower() in r.best["name"].lower())


def _preamble_ok(iq, fs, fc, want, bank) -> bool:
    res = identify(iq, fs, rf_center_hz=fc, top=1, preamble_bank=bank)
    return any(r.preamble is not None and want.lower() in r.preamble.name.lower()
               for r in res)

In [ ]:
def test_cfo_robustness():
    print("=" * 78)
    print("5. Carrier-offset robustness at 0 dB SNR (SiK, 250 kbps)")
    bank = PreambleBank(p_fa=1e-6)
    tpl = sik_preamble(250e3, 60e3)
    ok = True
    for ppm in (0, 2, 5, 10, 20, 40):
        cfo = ppm * 915e6 / 1e6
        good = 0
        for _ in range(6):
            iq = frame_gfsk(tpl, 2e6, 600, 0.0, 1e-3, cfo_hz=cfo)
            good += bool(_preamble_ok(iq, 2e6, 915e6, "SiK", bank))
        span = tpl.cfo_span
        inside = "in grid" if abs(cfo) <= span else "OUTSIDE grid"
        print(f"   {ppm:3d} ppm = {cfo/1e3:+7.1f} kHz  ({inside:12s})  {good}/6 detected")
        if abs(cfo) <= span and good < 6:
            ok = False
    print(f"   -> {'OK' if ok else 'FAIL'} (grid spans +/-{tpl.cfo_span/1e3:.0f} kHz)")
    return ok

In [ ]:
_ = test_cfo_robustness()

### The headline: measured SNR floors

Lowest SNR at which the protocol is identified on every trial. Published values
used 10 trials over a fine grid; with `QUICK = True` this runs 3 over a coarse
one and will land within a few dB.

| Protocol | Blind | Preamble | Gain | $N$ | Theory (4) |
|---|---|---|---|---|---|
| SiK/MAVLink 250 kbps | +25 dB | **−6 dB** | +31 dB | 224 | −9.4 dB |
| BLE 1M advertising | +25 dB | **−6 dB** | +31 dB | 320 | −11.1 dB |
| LoRa BW125 SF7 | +20 dB | **−18 dB** | +38 dB | 4096 | −22.5 dB |
| POCSAG 1200 bps | +25 dB | **−18 dB** | +43 dB | 3413 | −21.7 dB |

Measured floors sit 3–7 dB above theory, and the reasons are known rather than
mysterious: GFSK template mismatch from Gaussian ISI at the preamble boundary,
frequency-grid quantisation, and an all-trials-must-pass criterion that is
stricter than marginal detection. The *ordering* and the *scaling with $N$* both
follow the prediction.

In [ ]:
from iq_protocol_id import identify as _ident   # ensure the notebook's identify is used
def measure_floor(make, fs, fc, want, bank, mode, grid, trials):
    floor = None
    for snr in grid:
        good = sum(bool(_blind_ok(make(snr), fs, fc, want) if mode == "blind"
                        else _preamble_ok(make(snr), fs, fc, want, bank))
                   for _ in range(trials))
        if good == trials:
            floor = snr
        else:
            break
    return floor

bank = PreambleBank(p_fa=1e-6)
trials = 3 if QUICK else 10
grid = ([25, 10, 0, -6, -12, -18, -24] if QUICK
        else [30, 25, 20, 15, 10, 6, 3, 0, -3, -6, -9, -12, -15, -18, -21, -24])

sik = sik_preamble(250e3, 60e3)
lora_t = lora_preamble(125e3, 7)
poc = pocsag_preamble(1200.0, 4.5e3)
suite = [
    ("SiK/MAVLink 250 kbps", 2e6, 915e6, "SiK", sik,
     lambda s: frame_gfsk(sik, 2e6, 600, s, 1e-3)),
    ("BLE 1M advertising", 8e6, 2.44e9, "Bluetooth LE 1M", BLE_ADV_1M,
     lambda s: frame_gfsk(BLE_ADV_1M, 8e6, 300, s, 200e-6)),
    ("LoRa BW125 SF7", 500e3, 868e6, "LoRa", lora_t,
     lambda s: frame_lora(500e3, 125e3, 7, 20, s, 4e-3)),
    ("POCSAG 1200 bps", 64e3, 148e6, "POCSAG", poc,
     lambda s: frame_gfsk(poc, 64e3, 300, s, 20e-3)),
]

print(f"{'protocol':24s} {'blind':>8} {'preamble':>10} {'gain':>8} {'N':>7} {'theory':>9}")
for label, fs_, fc_, want, tpl, make in suite:
    fb = measure_floor(make, fs_, fc_, want, bank, "blind", grid, trials)
    fp = measure_floor(make, fs_, fc_, want, bank, "preamble", grid, trials)
    # N as the bank actually sees it, after its own channel decimation
    _, fsd, _dd = bank._decimate(np.zeros(4096, np.complex64), fs_, tpl)
    n = tpl.synth(fsd).size
    g = f"{fb - fp:+d} dB" if None not in (fb, fp) else "—"
    print(f"{label:24s} {str(fb)+' dB':>8} {str(fp)+' dB':>10} {g:>8} {n:7d} "
          f"{snr_floor_db(n, 10000):+8.1f} dB")

### End to end, at an SNR where blind analysis has no chance

The same capture, with and without the bank.

In [ ]:
def test_end_to_end():
    print("=" * 78)
    print("6. End-to-end identify() at -9 dB SNR")
    bank = PreambleBank(p_fa=1e-6)
    tpl = sik_preamble(250e3, 60e3)
    iq = frame_gfsk(tpl, 2e6, 600, -9.0, 1e-3, cfo_hz=4.2e3)
    print("   --- without the bank ---")
    print(format_report(identify(iq, 2e6, rf_center_hz=915e6, top=2)))
    print("   --- with the bank ---")
    print(format_report(identify(iq, 2e6, rf_center_hz=915e6, top=2,
                                 preamble_bank=bank)))
    return True

In [ ]:
_ = test_end_to_end()

# Part VI

## 26. Limitations

Honest accounting of what this does not do.

### Constants recalled, not read
Every specification constant here — the BLE access address and its bit order,
the ADS-B pulse positions, the Si4432 sync word, the protocol database's
bandwidth and rate ranges — comes from memory of the standards rather than from
the documents. §22 explains why the Zigbee chip sequence was omitted rather than
guessed, and the same skepticism should be applied to everything else before
trusting a real capture. Self-consistent tests **cannot** catch a wrong constant.

### The blind path
- **Band information is close to essential.** Without `rf_center_hz`, DMR and
  P25 are not separable, and SiK / Z-Wave / Wi-SUN 2-FSK overlap heavily — the
  scorer honestly reports several at 100%.
- **Ideal Nyquist shaping ($\alpha = 0$) has no recoverable symbol rate** by the
  cyclostationary method (§12). The spectral line genuinely vanishes.
- **Deviation estimates under-read** for heavily shaped GFSK at low samples per
  symbol: the DMR case recovers 4 tones but spacing biased by ISI.
- **Only AWGN is modelled.** No multipath, no fading, no adjacent-channel
  interference, no phase noise, no IQ imbalance or DC offset. Real captures have
  all of these, and frequency-selective fading in particular will break the
  bandwidth measurements that carry most of the classifier's information.
- **The "FFT size" from §10 is sometimes a training-sequence period.**
- **ADS-B vs Mode A/C/S replies are only partly separable** (2 of 12 seeds
  misclassify). Both sit at 1090 MHz with the same modulation and bandwidth,
  differing essentially in frame length; when the run-length chip-rate estimate
  wanders, duration alone is not always enough.

### Methodology
- **Single-seed test results are not measurements.** §18 documents four defects
  that a fixed seed concealed, including one that made six of eight cases return
  silently empty results. Any pass rate quoted here without a seed count should
  be distrusted, including the Part V floors — those used 10 trials per SNR
  point, which pins the floor to a few dB, not exactly.
- **Swallowing exceptions hides bugs.** The `except ValueError` in `identify`,
  intended for bursts too short to analyse, absorbed a genuine `numpy`
  broadcasting error for some time. Narrow the exception or re-raise unexpected
  ones.

### The preamble path
- **Only covers protocols with fixed preambles.** Anything negotiated
  (Bluetooth Classic access codes derive from the piconet LAP) needs the
  parameter before a template can be built.
- **Concurrent overlapping signals:** NMS keeps the stronger one (§24).
- **Cross-talk is bounded empirically, not analytically.** Equation (1) covers
  noise; the mutual coherence between templates is not characterised, and adding
  templates raises cross-talk in a way this suite does not measure.
- **CFO span is a fixed per-template guess.** Outside it, detection fails
  silently.
- **No Doppler or timing drift.** A long template assumes a stationary offset
  over its whole duration; the POCSAG template spans 107 ms, over which real
  oscillator drift is not negligible.

### Natural extensions
Demodulate after a hit (timing and CFO both fall out of the correlator, which is
most of acquisition); estimate the noise floor per-channel rather than
per-record; add a proper cyclic spectral correlation surface for blind symbol
rate at $\alpha = 0$; and replace the per-template CFO span with an offset
estimate from the coarse spectral centre.

## Summary

The classifier is interpretable by construction: every score decomposes into
per-feature terms that trace back to a derivation, so a wrong answer identifies
which measurement disagreed rather than requiring the model to be retrained.

The result worth carrying away is the division of labour. Blind features are
general — they classify anything, including protocols not in the database — but
are fundamentally limited to about 20 dB because they are second-order
statistics. A matched filter is narrow, requiring a documented preamble, but
converts template length into processing gain at the exponential rate of
equation (1), reaching −18 dB. Neither replaces the other: the correlator
supplies identification where it applies, and the blind features supply the
measurements it cannot produce and the corroboration that catches its
cross-talk.